<a href="https://colab.research.google.com/github/ariakiani2004/SNN-Surrogate-Gradient-MNIST/blob/main/SNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Fri Oct  9 19:11:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%cd /content

from google.colab import files
uploaded = files.upload()

/content


Saving environment_check.py to environment_check.py


In [ ]:
!python /content/environment_check.py \
  --device cuda:0 \
  --output /content/stage1/environment_report.json

[PASS] import numpy: 2.1.3
[PASS] import torch: 2.11.0+cu130
[PASS] import torchvision: 0.26.0+cu130
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
[PASS] import ultralytics: 8.4.173
[PASS] import cv2: 5.0.0.93
[PASS] import PIL: 11.3.0
[PASS] import yaml: 6.0.3
[PASS] import pycocotools: 2.0.11
[PASS] NumPy random / ABI: Compiled random extension works
[PASS] NumPy / PyTorch bridge: NumPy -> torch -> NumPy works
[PASS] CUDA allocation: {'name': 'Tesla T4', 'cuda_runtime': '13.0', 'count': 1, 'memory_gib': 14.56, 'compute_capability': [7, 5]}
[PASS] GPU forward / backward / optimizer: {'precision': 'FP32', 'loss': 0.3512011468410492, 'weights_updated': True}
[PASS] torchvision compiled operator: Compiled torchvision operation works on cuda:0
[P

In [ ]:
!python -m pip install "ultralytics==8.4.173"

In [ ]:
!python /content/environment_check.py \
  --device cuda:0 \
  --output /content/stage1/environment_report.json

[PASS] import numpy: 2.1.3
[PASS] import torch: 2.11.0+cu130
[PASS] import torchvision: 0.26.0+cu130
[PASS] import ultralytics: 8.4.173
[PASS] import cv2: 5.0.0.93
[PASS] import PIL: 11.3.0
[PASS] import yaml: 6.0.3
[PASS] import pycocotools: 2.0.11
[PASS] NumPy random / ABI: Compiled random extension works
[PASS] NumPy / PyTorch bridge: NumPy -> torch -> NumPy works
[PASS] CUDA allocation: {'name': 'Tesla T4', 'cuda_runtime': '13.0', 'count': 1, 'memory_gib': 14.56, 'compute_capability': [7, 5]}
[PASS] GPU forward / backward / optimizer: {'precision': 'FP32', 'loss': 0.3512011468410492, 'weights_updated': True}
[PASS] torchvision compiled operator: Compiled torchvision operation works on cuda:0
[PASS] pycocotools / NumPy ABI: COCO imports and compiled mask round trip work
[PASS] Ultralytics module imports: C3k2 and Detect are available; project trainer compatibility checked later

STAGE 1: PASS
Report: /content/stage1/environment_report.json
Recorded versions describe this environment

In [ ]:
from pathlib import Path

PROJECT = Path("/content/yolo26_lif")
PROJECT.mkdir(parents=True, exist_ok=True)

for folder in ["configs", "runs/stage2"]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

In [ ]:
from ultralytics.data.utils import check_det_dataset

dataset = check_det_dataset("coco8.yaml")

print("Dataset:", dataset["path"])
print("Train:", dataset["train"])
print("Val:", dataset["val"])
print("Classes:", dataset["nc"])

assert dataset["nc"] == 80


WARNING ⚠️ Dataset 'coco8.yaml' images not found, missing path '/content/datasets/coco8/images/val'
Unzipping /content/datasets/coco8.zip to /content/datasets/coco8...: 100% ━━━━━━━━━━━━ 25/25 3.4Kfiles/s 0.0s
Dataset download success ✅ (0.5s), saved to /content/datasets

Dataset: /content/datasets/coco8
Train: /content/datasets/coco8/images/train
Val: /content/datasets/coco8/images/val
Classes: 80


In [ ]:
from pathlib import Path
from PIL import Image
import hashlib
import json
import math

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
report = {}
hashes = {}

for split in ["train", "val"]:
    image_dir = Path(dataset[split])
    assert image_dir.is_dir(), f"Missing directory: {image_dir}"

    # ساختار استاندارد COCO8: images/train و labels/train
    label_dir = image_dir.parent.parent / "labels" / image_dir.name
    images = sorted(
        p for p in image_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    )
    assert images, f"No images in {image_dir}"

    instances = 0
    split_hashes = set()

    for image_path in images:
        with Image.open(image_path) as image:
            image.verify()

        split_hashes.add(
            hashlib.sha256(image_path.read_bytes()).hexdigest()
        )

        label_path = label_dir / f"{image_path.stem}.txt"
        assert label_path.is_file(), f"Missing label: {label_path}"

        for line_number, line in enumerate(
            label_path.read_text().splitlines(), start=1
        ):
            if not line.strip():
                continue

            fields = line.split()
            assert len(fields) == 5, (
                f"Invalid detection label: {label_path}:{line_number}"
            )

            cls, x, y, w, h = map(float, fields)
            assert all(math.isfinite(v) for v in [cls, x, y, w, h])
            assert cls.is_integer() and 0 <= cls < dataset["nc"]
            assert 0 <= x <= 1 and 0 <= y <= 1
            assert 0 < w <= 1 and 0 < h <= 1
            instances += 1

    hashes[split] = split_hashes
    report[split] = {
        "image_directory": str(image_dir),
        "label_directory": str(label_dir),
        "images": len(images),
        "instances": instances,
    }

assert not hashes["train"].intersection(hashes["val"]), (
    "Identical image files found in train and val"
)

output = PROJECT / "runs/stage2/dataset_report.json"
output.write_text(json.dumps(report, indent=2), encoding="utf-8")

print(json.dumps(report, indent=2))
print("\nSTAGE 2: PASS — COCO8 smoke-test data")
print("Report:", output)

{
  "train": {
    "image_directory": "/content/datasets/coco8/images/train",
    "label_directory": "/content/datasets/coco8/labels/train",
    "images": 4,
    "instances": 13
  },
  "val": {
    "image_directory": "/content/datasets/coco8/images/val",
    "label_directory": "/content/datasets/coco8/labels/val",
    "images": 4,
    "instances": 17
  }
}

STAGE 2: PASS — COCO8 smoke-test data
Report: /content/yolo26_lif/runs/stage2/dataset_report.json


In [ ]:
import yaml

data_config = {
    "path": str(Path(dataset["path"]).resolve()),
    "train": str(Path(dataset["train"]).resolve()),
    "val": str(Path(dataset["val"]).resolve()),
    "names": dataset["names"],
}

config_path = PROJECT / "configs/coco8_local.yaml"

config_path.write_text(
    yaml.safe_dump(data_config, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)

print("Config:", config_path)

Config: /content/yolo26_lif/configs/coco8_local.yaml


In [ ]:
from pathlib import Path
from google.colab import files
import shutil

PROJECT = Path("/content/yolo26_lif")
WEIGHTS = PROJECT / "weights/yolo26n.pt"
DATA = PROJECT / "configs/coco8_local.yaml"
OUTPUT = PROJECT / "runs/stage3_ann"

WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

assert DATA.is_file(), f"Missing dataset config: {DATA}"

if not WEIGHTS.is_file():
    print("Upload the original pretrained yolo26n.pt")
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.endswith(".pt")]
    assert len(candidates) == 1, "Upload exactly one .pt file"
    shutil.copyfile(candidates[0], WEIGHTS)

print("Weights:", WEIGHTS)

Upload the original pretrained yolo26n.pt


Saving yolo26n.pt to yolo26n (1).pt
Weights: /content/yolo26_lif/weights/yolo26n.pt


In [ ]:
from ultralytics import YOLO
import hashlib
import json
import torch
import ultralytics

ann = YOLO(str(WEIGHTS), task="detect")

# جلوگیری از استفادهٔ تصادفی از مدل تبدیل‌شدهٔ پروژه
assert not hasattr(ann.model, "snn_config"), (
    "This checkpoint contains SNN metadata. Use the original ANN weights."
)

module_rows = []

for index, module in enumerate(ann.model.model):
    row = {
        "index": index,
        "type": type(module).__name__,
        "from": getattr(module, "f", None),
        "parameters": sum(p.numel() for p in module.parameters()),
    }
    module_rows.append(row)
    print(row)

c3k2_indices = [
    row["index"] for row in module_rows
    if row["type"] == "C3k2"
]

assert c3k2_indices, "No C3k2 modules found; inspect the checkpoint."
print("\nC3k2 indices:", c3k2_indices)

baseline = {
    "weights": str(WEIGHTS),
    "weights_sha256": hashlib.sha256(WEIGHTS.read_bytes()).hexdigest(),
    "data": str(DATA),
    "torch_version": torch.__version__,
    "ultralytics_version": ultralytics.__version__,
    "parameters_before_validation": sum(
        p.numel() for p in ann.model.parameters()
    ),
    "modules": module_rows,
    "c3k2_indices": c3k2_indices,
}

{'index': 0, 'type': 'Conv', 'from': -1, 'parameters': 464}
{'index': 1, 'type': 'Conv', 'from': -1, 'parameters': 4672}
{'index': 2, 'type': 'C3k2', 'from': -1, 'parameters': 6640}
{'index': 3, 'type': 'Conv', 'from': -1, 'parameters': 36992}
{'index': 4, 'type': 'C3k2', 'from': -1, 'parameters': 26080}
{'index': 5, 'type': 'Conv', 'from': -1, 'parameters': 147712}
{'index': 6, 'type': 'C3k2', 'from': -1, 'parameters': 87040}
{'index': 7, 'type': 'Conv', 'from': -1, 'parameters': 295424}
{'index': 8, 'type': 'C3k2', 'from': -1, 'parameters': 346112}
{'index': 9, 'type': 'SPPF', 'from': -1, 'parameters': 164608}
{'index': 10, 'type': 'C2PSA', 'from': -1, 'parameters': 249728}
{'index': 11, 'type': 'Upsample', 'from': -1, 'parameters': 0}
{'index': 12, 'type': 'Concat', 'from': [-1, 6], 'parameters': 0}
{'index': 13, 'type': 'C3k2', 'from': -1, 'parameters': 119808}
{'index': 14, 'type': 'Upsample', 'from': -1, 'parameters': 0}
{'index': 15, 'type': 'Concat', 'from': [-1, 4], 'parameter

In [ ]:
shapes = {}
hooks = []

def make_hook(index):
    def hook(module, inputs, output):
        shapes[str(index)] = {
            "input_shape": list(inputs[0].shape),
            "output_shape": list(output.shape),
        }
    return hook

try:
    for index in c3k2_indices:
        hooks.append(
            ann.model.model[index].register_forward_hook(make_hook(index))
        )

    ann.model.to("cuda:0").eval()

    with torch.inference_mode():
        dummy = torch.zeros(1, 3, 320, 320, device="cuda:0")
        _ = ann.model(dummy)

    torch.cuda.synchronize()

finally:
    for hook in hooks:
        hook.remove()

assert len(shapes) == len(c3k2_indices)
baseline["c3k2_shapes"] = shapes

print(json.dumps(shapes, indent=2))

{
  "2": {
    "input_shape": [
      1,
      32,
      80,
      80
    ],
    "output_shape": [
      1,
      64,
      80,
      80
    ]
  },
  "4": {
    "input_shape": [
      1,
      64,
      40,
      40
    ],
    "output_shape": [
      1,
      128,
      40,
      40
    ]
  },
  "6": {
    "input_shape": [
      1,
      128,
      20,
      20
    ],
    "output_shape": [
      1,
      128,
      20,
      20
    ]
  },
  "8": {
    "input_shape": [
      1,
      256,
      10,
      10
    ],
    "output_shape": [
      1,
      256,
      10,
      10
    ]
  },
  "13": {
    "input_shape": [
      1,
      384,
      20,
      20
    ],
    "output_shape": [
      1,
      128,
      20,
      20
    ]
  },
  "16": {
    "input_shape": [
      1,
      256,
      40,
      40
    ],
    "output_shape": [
      1,
      64,
      40,
      40
    ]
  },
  "19": {
    "input_shape": [
      1,
      192,
      20,
      20
    ],
    "output_shape": [
      1,
    

In [ ]:
metrics = ann.val(
    data=str(DATA),
    split="val",
    imgsz=320,
    batch=2,
    device=0,
    workers=2,
    conf=0.001,
    iou=0.7,
    augment=False,
    plots=True,
    project=str(OUTPUT),
    name="validation",
    exist_ok=True,
)

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 120 layers, 2,408,932 parameters, 0 gradients, 5.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1357.0±283.8 MB/s, size: 54.0 KB)
val: Scanning /content/datasets/coco8/labels/val... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 1.5Kit/s 0.0s
val: New cache created: /content/datasets/coco8/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 5.0it/s 0.4s
                   all          4         17      0.666      0.617       0.78      0.605
                person          3         10      0.258        0.2      0.205       0.13
                   dog          1          1      0.675          1      0.995      0.697
                 horse          1          2      0.677          1      0.995      0.519
              elephant          1          2      0.817        0.5      0.495  

In [ ]:
baseline["evaluation"] = {
    "imgsz": 320,
    "batch": 2,
    "split": "val",
    "conf": 0.001,
    "iou": 0.7,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "speed_ms_per_image": {
        key: float(value)
        for key, value in metrics.speed.items()
    },
    "save_dir": str(metrics.save_dir),
}

report_path = OUTPUT / "ann_baseline.json"
report_path.write_text(
    json.dumps(baseline, indent=2),
    encoding="utf-8",
)

print(json.dumps(baseline["evaluation"], indent=2))
print("\nSTAGE 3: PASS — ANN baseline recorded")
print("Report:", report_path)

{
  "imgsz": 320,
  "batch": 2,
  "split": "val",
  "conf": 0.001,
  "iou": 0.7,
  "precision": 0.6657676002260294,
  "recall": 0.6166666666666667,
  "mAP50": 0.7799920510718633,
  "mAP50_95": 0.6052667022812892,
  "speed_ms_per_image": {
    "preprocess": 0.8208322500138365,
    "inference": 27.509076000001187,
    "loss": 0.004313999994565165,
    "postprocess": 1.6312222500118878
  },
  "save_dir": "/content/yolo26_lif/runs/stage3_ann/validation"
}

STAGE 3: PASS — ANN baseline recorded
Report: /content/yolo26_lif/runs/stage3_ann/ann_baseline.json


In [ ]:
import json
from pathlib import Path

report_path = Path(
    "/content/yolo26_lif/runs/stage3_ann/ann_baseline.json"
)
saved = json.loads(report_path.read_text())

expected = {str(i) for i in saved["c3k2_indices"]}
recorded = saved.get("c3k2_shapes", {})

assert set(recorded) == expected, "C3k2 shapes are incomplete"

print(json.dumps(recorded, indent=2))
print("\nSTAGE 3 STRUCTURE: PASS")

{
  "2": {
    "input_shape": [
      1,
      32,
      80,
      80
    ],
    "output_shape": [
      1,
      64,
      80,
      80
    ]
  },
  "4": {
    "input_shape": [
      1,
      64,
      40,
      40
    ],
    "output_shape": [
      1,
      128,
      40,
      40
    ]
  },
  "6": {
    "input_shape": [
      1,
      128,
      20,
      20
    ],
    "output_shape": [
      1,
      128,
      20,
      20
    ]
  },
  "8": {
    "input_shape": [
      1,
      256,
      10,
      10
    ],
    "output_shape": [
      1,
      256,
      10,
      10
    ]
  },
  "13": {
    "input_shape": [
      1,
      384,
      20,
      20
    ],
    "output_shape": [
      1,
      128,
      20,
      20
    ]
  },
  "16": {
    "input_shape": [
      1,
      256,
      40,
      40
    ],
    "output_shape": [
      1,
      64,
      40,
      40
    ]
  },
  "19": {
    "input_shape": [
      1,
      192,
      20,
      20
    ],
    "output_shape": [
      1,
    

In [ ]:
from pathlib import Path

PROJECT = Path("/content/yolo26_lif")
MODULE_DIR = PROJECT / "lif26"

MODULE_DIR.mkdir(parents=True, exist_ok=True)
(MODULE_DIR / "__init__.py").touch(exist_ok=True)

%cd /content/yolo26_lif

/content/yolo26_lif


In [ ]:
%%writefile /content/yolo26_lif/lif26/neuron.py
import math

import torch
from torch import nn


class _SurrogateSpike(torch.autograd.Function):
    """Binary forward; fast-sigmoid surrogate backward."""

    @staticmethod
    def forward(ctx, voltage_minus_threshold, slope):
        ctx.save_for_backward(voltage_minus_threshold)
        ctx.slope = slope
        return (voltage_minus_threshold >= 0).to(
            voltage_minus_threshold.dtype
        )

    @staticmethod
    def backward(ctx, grad_output):
        (distance,) = ctx.saved_tensors
        surrogate = (
            1.0 + ctx.slope * distance.abs()
        ).square().reciprocal()

        return grad_output * surrogate, None


class LIF(nn.Module):
    """One LIF time step with explicit state.

    Input:
        current: floating-point tensor.
        membrane: previous FP32 state, or None for a new sequence.

    Output:
        spike: binary tensor in the input dtype.
        membrane: updated FP32 state.

    Pass membrane=None for each independent input sequence.
    Keep returned states attached during temporal training.
    """

    def __init__(self, beta=0.5, threshold=1.0, surrogate_slope=10.0):
        super().__init__()

        if not math.isfinite(beta) or not 0.0 <= beta < 1.0:
            raise ValueError("beta must satisfy 0 <= beta < 1")
        if not math.isfinite(threshold) or threshold <= 0.0:
            raise ValueError("threshold must be positive and finite")
        if not math.isfinite(surrogate_slope) or surrogate_slope <= 0.0:
            raise ValueError("surrogate_slope must be positive and finite")

        self.beta = float(beta)
        self.threshold = float(threshold)
        self.surrogate_slope = float(surrogate_slope)

    def forward(self, current, membrane=None):
        if not current.is_floating_point():
            raise TypeError("current must be a floating-point tensor")

        # Preserve FP32 membrane arithmetic under mixed precision.
        with torch.autocast(device_type=current.device.type, enabled=False):
            current_fp32 = current.float()

            if membrane is None:
                membrane = torch.zeros_like(current_fp32)
            else:
                if membrane.shape != current.shape:
                    raise ValueError("membrane and current shapes must match")
                if membrane.device != current.device:
                    raise ValueError("membrane and current devices must match")
                if membrane.dtype != torch.float32:
                    raise TypeError("membrane must have dtype torch.float32")

            voltage = self.beta * membrane + current_fp32

            spike_fp32 = _SurrogateSpike.apply(
                voltage - self.threshold,
                self.surrogate_slope,
            )

            # Detach only the reset spike, not the temporal state.
            next_membrane = (
                voltage - self.threshold * spike_fp32.detach()
            )

        return spike_fp32.to(current.dtype), next_membrane

    def extra_repr(self):
        return (
            f"beta={self.beta}, threshold={self.threshold}, "
            f"surrogate_slope={self.surrogate_slope}, reset=soft"
        )

Writing /content/yolo26_lif/lif26/neuron.py


In [ ]:
import json
import torch

from lif26.neuron import LIF

assert torch.cuda.is_available()
device = torch.device("cuda:0")

results = {}

# 1. Exact recurrence and soft reset
lif = LIF(beta=0.5, threshold=1.0).to(device)

membrane = None
spikes = []
states = []

for _ in range(4):
    spike, membrane = lif(
        torch.tensor([0.6], device=device),
        membrane,
    )
    spikes.append(spike.item())
    states.append(membrane.item())

torch.testing.assert_close(
    torch.tensor(spikes),
    torch.tensor([0.0, 0.0, 1.0, 0.0]),
)

torch.testing.assert_close(
    torch.tensor(states),
    torch.tensor([0.6, 0.9, 0.05, 0.625]),
    rtol=1e-5,
    atol=1e-6,
)

results["recurrence_and_soft_reset"] = "PASS"

# 2. Exactly at threshold: spike = 1
spike, membrane = lif(torch.tensor([1.0], device=device))
assert spike.item() == 1.0
assert membrane.item() == 0.0

results["threshold_boundary"] = "PASS"

# 3. Independent sequences start from zero state
current = torch.full((2, 8, 8, 8), 0.6, device=device)

spike_a, state_a = lif(current, None)
spike_b, state_b = lif(current, None)

torch.testing.assert_close(spike_a, spike_b)
torch.testing.assert_close(state_a, state_b)

assert spike_a.count_nonzero().item() == 0

results["independent_sequences"] = "PASS"

# 4. Surrogate gradients flow through time
currents = [
    torch.tensor([0.6], device=device, requires_grad=True)
    for _ in range(3)
]

membrane = None
for current in currents:
    final_spike, membrane = lif(current, membrane)

final_spike.sum().backward()

assert final_spike.item() == 1.0

for current in currents:
    assert current.grad is not None
    assert torch.isfinite(current.grad).all().item()
    assert current.grad.abs().sum().item() > 0

# At t=3, voltage-threshold = 0.05.
last_gradient = 1.0 / (1.0 + 10.0 * 0.05) ** 2
expected_gradients = [
    last_gradient * 0.5**2,
    last_gradient * 0.5,
    last_gradient,
]

torch.testing.assert_close(
    torch.stack([x.grad.squeeze() for x in currents]),
    torch.tensor(expected_gradients, device=device),
    rtol=1e-4,
    atol=1e-6,
)

results["temporal_surrogate_gradient"] = "PASS"

# 5. FP16 spikes with FP32 membrane under autocast
current_fp16 = torch.full(
    (2, 8, 8, 8),
    0.6,
    device=device,
    dtype=torch.float16,
    requires_grad=True,
)

with torch.autocast(device_type="cuda", dtype=torch.float16):
    spike_fp16, state_fp32 = lif(current_fp16)

assert spike_fp16.dtype == torch.float16
assert state_fp32.dtype == torch.float32
assert torch.all(
    (spike_fp16 == 0) | (spike_fp16 == 1)
).item()

spike_fp16.float().sum().backward()

assert torch.isfinite(state_fp32).all().item()
assert current_fp16.grad is not None
assert torch.isfinite(current_fp16.grad).all().item()
assert current_fp16.grad.abs().sum().item() > 0

results["mixed_precision"] = "PASS"

torch.cuda.synchronize()

output_dir = PROJECT / "runs/stage4_lif"
output_dir.mkdir(parents=True, exist_ok=True)

report = {
    "status": "PASS",
    "device": torch.cuda.get_device_name(0),
    "neuron": "LIF",
    "beta": lif.beta,
    "threshold": lif.threshold,
    "surrogate_slope": lif.surrogate_slope,
    "reset": "soft",
    "membrane_dtype": "float32",
    "checks": results,
}

report_path = output_dir / "lif_report.json"
report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print(json.dumps(report, indent=2))
print("\nSTAGE 4: PASS")
print("Report:", report_path)

{
  "status": "PASS",
  "device": "Tesla T4",
  "neuron": "LIF",
  "beta": 0.5,
  "threshold": 1.0,
  "surrogate_slope": 10.0,
  "reset": "soft",
  "membrane_dtype": "float32",
  "checks": {
    "recurrence_and_soft_reset": "PASS",
    "threshold_boundary": "PASS",
    "independent_sequences": "PASS",
    "temporal_surrogate_gradient": "PASS",
    "mixed_precision": "PASS"
  }
}

STAGE 4: PASS
Report: /content/yolo26_lif/runs/stage4_lif/lif_report.json


In [ ]:
%cd /content/yolo26_lif

from pathlib import Path

PROJECT = Path("/content/yolo26_lif")
assert (PROJECT / "lif26/neuron.py").is_file()

(PROJECT / "runs/stage5_blocks").mkdir(
    parents=True,
    exist_ok=True,
)

/content/yolo26_lif


In [ ]:
%%writefile /content/yolo26_lif/lif26/blocks.py
import torch
from torch import nn

from .neuron import LIF


class SpikeConv(nn.Module):
    """Conv + BatchNorm + one explicit-state LIF step."""

    def __init__(
        self,
        c1,
        c2,
        k=1,
        s=1,
        p=None,
        g=1,
        d=1,
        beta=0.5,
        threshold=1.0,
        surrogate_slope=10.0,
    ):
        super().__init__()

        if not isinstance(k, int) or k <= 0 or k % 2 == 0:
            raise ValueError("k must be a positive odd integer")
        if s < 1 or d < 1 or g < 1:
            raise ValueError("s, d and g must be positive")
        if c1 % g != 0 or c2 % g != 0:
            raise ValueError("Channels must be divisible by groups")

        if p is None:
            p = d * (k - 1) // 2

        self.conv = nn.Conv2d(
            c1,
            c2,
            kernel_size=k,
            stride=s,
            padding=p,
            groups=g,
            dilation=d,
            bias=False,
        )

        # Match the usual Ultralytics BN configuration.
        self.bn = nn.BatchNorm2d(
            c2,
            eps=1e-3,
            momentum=0.03,
        )

        self.lif = LIF(
            beta=beta,
            threshold=threshold,
            surrogate_slope=surrogate_slope,
        )

    def forward_current(self, x):
        """Analog current before LIF; used for pre-fire residual."""
        return self.bn(self.conv(x))

    def forward(self, x, membrane=None):
        current = self.forward_current(x)
        return self.lif(current, membrane)


class SpikeBottleneck(nn.Module):
    """Two-convolution bottleneck with pre-fire residual.

    state:
        None, or (membrane_1, membrane_2).

    Returns:
        binary output, updated state tuple.

    Residual is enabled only when shortcut=True and c1 == c2.
    """

    def __init__(
        self,
        c1,
        c2,
        shortcut=True,
        g=1,
        k=(3, 3),
        e=0.5,
        beta=0.5,
        threshold=1.0,
        surrogate_slope=10.0,
    ):
        super().__init__()

        if not 0 < e <= 1:
            raise ValueError("e must satisfy 0 < e <= 1")
        if not isinstance(k, (tuple, list)) or len(k) != 2:
            raise ValueError("k must contain two kernel sizes")

        hidden = int(c2 * e)
        if hidden < 1:
            raise ValueError("Hidden channel count must be positive")

        lif_options = {
            "beta": beta,
            "threshold": threshold,
            "surrogate_slope": surrogate_slope,
        }

        self.cv1 = SpikeConv(
            c1, hidden, k=k[0], **lif_options
        )
        self.cv2 = SpikeConv(
            hidden, c2, k=k[1], g=g, **lif_options
        )

        self.add = bool(shortcut and c1 == c2)

    def forward(self, x, state=None):
        if state is None:
            membrane_1, membrane_2 = None, None
        else:
            if not isinstance(state, tuple) or len(state) != 2:
                raise TypeError(
                    "state must be None or a two-element tuple"
                )
            membrane_1, membrane_2 = state

        spike_1, next_membrane_1 = self.cv1(
            x,
            membrane_1,
        )

        current_2 = self.cv2.forward_current(spike_1)

        if self.add:
            # Sum before thresholding. Keep the sum in FP32.
            current_2 = current_2.float() + x.float()

        spike_2, next_membrane_2 = self.cv2.lif(
            current_2,
            membrane_2,
        )

        # Preserve the feature dtype for following convolutions.
        spike_2 = spike_2.to(spike_1.dtype)

        return spike_2, (
            next_membrane_1,
            next_membrane_2,
        )

Writing /content/yolo26_lif/lif26/blocks.py


In [ ]:
import json
import torch

from lif26.blocks import SpikeConv, SpikeBottleneck

assert torch.cuda.is_available()
device = torch.device("cuda:0")
torch.manual_seed(42)

checks = {}


def assert_binary(x):
    assert torch.isfinite(x).all().item()
    assert torch.all((x == 0) | (x == 1)).item()


# SpikeConv: preserve spatial dimensions
conv = SpikeConv(16, 32, k=3).to(device).eval()
x = torch.randn(2, 16, 20, 20, device=device)

with torch.no_grad():
    y, membrane = conv(x)

assert y.shape == (2, 32, 20, 20)
assert membrane.shape == y.shape
assert membrane.dtype == torch.float32
assert_binary(y)

checks["spikeconv_shape_and_binary_output"] = "PASS"


# SpikeConv: stride=2
downsample = SpikeConv(
    16, 32, k=3, s=2
).to(device).eval()

with torch.no_grad():
    y_down, _ = downsample(x)

assert y_down.shape == (2, 32, 10, 10)
assert_binary(y_down)

checks["spikeconv_stride"] = "PASS"


# Bottleneck with residual
block = SpikeBottleneck(
    16, 16, shortcut=True, e=0.5
).to(device).eval()

with torch.no_grad():
    y, state = block(x)

assert block.add is True
assert y.shape == x.shape
assert state[0].shape == (2, 8, 20, 20)
assert state[1].shape == x.shape
assert all(v.dtype == torch.float32 for v in state)
assert_binary(y)

checks["bottleneck_residual_shapes"] = "PASS"


# Different channels: residual must be disabled
projection = SpikeBottleneck(
    16, 32, shortcut=True, e=0.5
).to(device).eval()

with torch.no_grad():
    y_projection, _ = projection(x)

assert projection.add is False
assert y_projection.shape == (2, 32, 20, 20)
assert_binary(y_projection)

checks["bottleneck_channel_change"] = "PASS"

print(json.dumps(checks, indent=2))

{
  "spikeconv_shape_and_binary_output": "PASS",
  "spikeconv_stride": "PASS",
  "bottleneck_residual_shapes": "PASS",
  "bottleneck_channel_change": "PASS"
}


In [ ]:
residual_test = SpikeBottleneck(
    4, 4,
    shortcut=True,
    e=1.0,
    beta=0.5,
    threshold=1.0,
).to(device).eval()

with torch.no_grad():
    residual_test.cv2.conv.weight.zero_()
    residual_test.cv2.bn.bias.zero_()
    residual_test.cv2.bn.running_mean.zero_()

    current = torch.full(
        (1, 4, 4, 4),
        0.6,
        device=device,
    )

    state = None
    observed_spikes = []
    observed_membranes = []

    for _ in range(4):
        output, state = residual_test(current, state)
        observed_spikes.append(output[0, 0, 0, 0].item())
        observed_membranes.append(state[1][0, 0, 0, 0].item())

torch.testing.assert_close(
    torch.tensor(observed_spikes),
    torch.tensor([0.0, 0.0, 1.0, 0.0]),
)

torch.testing.assert_close(
    torch.tensor(observed_membranes),
    torch.tensor([0.6, 0.9, 0.05, 0.625]),
    rtol=1e-5,
    atol=1e-6,
)

checks["pre_fire_residual_and_temporal_state"] = "PASS"


# New sequence: previous states must not be reused.
with torch.no_grad():
    first_output, first_state = residual_test(current, None)
    second_output, second_state = residual_test(current, None)

torch.testing.assert_close(first_output, second_output)

for a, b in zip(first_state, second_state):
    torch.testing.assert_close(a, b)

assert first_output.count_nonzero().item() == 0

checks["independent_sequences"] = "PASS"

print("Residual and temporal-state tests: PASS")

Residual and temporal-state tests: PASS


In [ ]:
train_block = SpikeBottleneck(
    16, 16,
    shortcut=True,
    e=0.5,
).to(device).train()

optimizer = torch.optim.SGD(
    train_block.parameters(),
    lr=0.05,
)

x_train = torch.randn(
    2, 16, 20, 20,
    device=device,
    requires_grad=True,
)

before = {
    "cv1": train_block.cv1.conv.weight.detach().clone(),
    "cv2": train_block.cv2.conv.weight.detach().clone(),
}

optimizer.zero_grad(set_to_none=True)

state = None
outputs = []

for _ in range(4):
    output, state = train_block(x_train, state)
    outputs.append(output)

rate = torch.stack(outputs, dim=0).float().mean(dim=0)

# Synthetic target for checking that learning works.
loss = (rate - 0.25).square().mean()

assert torch.isfinite(loss).item()
loss.backward()

assert x_train.grad is not None
assert torch.isfinite(x_train.grad).all().item()
assert x_train.grad.abs().sum().item() > 0

for name, parameter in train_block.named_parameters():
    assert parameter.grad is not None, f"Missing gradient: {name}"
    assert torch.isfinite(parameter.grad).all().item(), (
        f"Non-finite gradient: {name}"
    )

for name, layer in [
    ("cv1", train_block.cv1),
    ("cv2", train_block.cv2),
]:
    assert layer.conv.weight.grad.abs().sum().item() > 0, (
        f"Zero convolution gradient: {name}"
    )

optimizer.step()

assert not torch.equal(
    before["cv1"],
    train_block.cv1.conv.weight.detach(),
)
assert not torch.equal(
    before["cv2"],
    train_block.cv2.conv.weight.detach(),
)

checks["temporal_backward_and_weight_updates"] = "PASS"

print("Loss:", loss.item())
print("Temporal backward and optimizer: PASS")

Loss: 0.15605956315994263
Temporal backward and optimizer: PASS


In [ ]:
amp_block = SpikeBottleneck(
    16, 16,
    shortcut=True,
    e=0.5,
).to(device).train()

x_amp = torch.randn(
    2, 16, 20, 20,
    device=device,
    requires_grad=True,
)

amp_block.zero_grad(set_to_none=True)

with torch.autocast(device_type="cuda", dtype=torch.float16):
    state = None
    outputs = []

    for _ in range(2):
        output, state = amp_block(x_amp, state)
        outputs.append(output)

# Compute test loss in FP32.
amp_rate = torch.stack(outputs).float().mean(dim=0)
amp_loss = (amp_rate - 0.25).square().mean()
amp_loss.backward()

assert output.dtype == torch.float16
assert_binary(output)
assert all(v.dtype == torch.float32 for v in state)
assert all(torch.isfinite(v).all().item() for v in state)

for name, parameter in amp_block.named_parameters():
    assert parameter.grad is not None, f"Missing gradient: {name}"
    assert torch.isfinite(parameter.grad).all().item(), (
        f"Non-finite gradient: {name}"
    )

for layer in [amp_block.cv1, amp_block.cv2]:
    assert layer.conv.weight.grad.abs().sum().item() > 0

checks["mixed_precision"] = "PASS"

print("Mixed precision: PASS")

Mixed precision: PASS


In [ ]:
output_dir = PROJECT / "runs/stage5_blocks"
checkpoint_path = output_dir / "spike_bottleneck_test.pt"

trained_eval = train_block.eval()

torch.save(
    trained_eval.state_dict(),
    checkpoint_path,
)

restored = SpikeBottleneck(
    16, 16,
    shortcut=True,
    e=0.5,
).to(device).eval()

restored.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True,
    )
)

with torch.no_grad():
    reference_output, reference_state = trained_eval(
        x_train.detach(), None
    )
    restored_output, restored_state = restored(
        x_train.detach(), None
    )

torch.testing.assert_close(reference_output, restored_output)

for a, b in zip(reference_state, restored_state):
    torch.testing.assert_close(a, b)

checks["checkpoint_round_trip"] = "PASS"

torch.cuda.synchronize()

report = {
    "status": "PASS",
    "device": torch.cuda.get_device_name(0),
    "blocks": ["SpikeConv", "SpikeBottleneck"],
    "beta": 0.5,
    "threshold": 1.0,
    "surrogate_slope": 10.0,
    "residual_position": "before_output_LIF",
    "membrane_dtype": "float32",
    "temporal_training_steps_tested": 4,
    "checks": checks,
}

report_path = output_dir / "blocks_report.json"

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print(json.dumps(report, indent=2))
print("\nSTAGE 5: PASS")
print("Report:", report_path)

{
  "status": "PASS",
  "device": "Tesla T4",
  "blocks": [
    "SpikeConv",
    "SpikeBottleneck"
  ],
  "beta": 0.5,
  "threshold": 1.0,
  "surrogate_slope": 10.0,
  "residual_position": "before_output_LIF",
  "membrane_dtype": "float32",
  "temporal_training_steps_tested": 4,
  "checks": {
    "spikeconv_shape_and_binary_output": "PASS",
    "spikeconv_stride": "PASS",
    "bottleneck_residual_shapes": "PASS",
    "bottleneck_channel_change": "PASS",
    "pre_fire_residual_and_temporal_state": "PASS",
    "independent_sequences": "PASS",
    "temporal_backward_and_weight_updates": "PASS",
    "mixed_precision": "PASS",
    "checkpoint_round_trip": "PASS"
  }
}

STAGE 5: PASS
Report: /content/yolo26_lif/runs/stage5_blocks/blocks_report.json


In [ ]:
%cd /content/yolo26_lif

from pathlib import Path

PROJECT = Path("/content/yolo26_lif")
WEIGHTS = PROJECT / "weights/yolo26n.pt"

for path in [
    PROJECT / "lif26/neuron.py",
    PROJECT / "lif26/blocks.py",
    WEIGHTS,
]:
    assert path.is_file(), f"Missing file: {path}"

OUTPUT = PROJECT / "runs/stage6_csp"
OUTPUT.mkdir(parents=True, exist_ok=True)

/content/yolo26_lif


In [ ]:
import json
import torch

from ultralytics import YOLO
from lif26.csp import SpikeC3k2
from lif26.neuron import LIF

assert torch.cuda.is_available()
device = torch.device("cuda:0")

source_yolo = YOLO(str(WEIGHTS), task="detect")
source_model = source_yolo.model.cpu().eval()

assert not hasattr(source_model, "snn_config"), (
    "Use the original ANN checkpoint"
)

indices = [
    index
    for index, module in enumerate(source_model.model)
    if type(module).__name__ == "C3k2"
]

assert indices == [2, 4, 6, 8, 13, 16, 19, 22], (
    f"Unexpected module indices: {indices}"
)

print("C3k2 indices:", indices)
print("\nStructure of module 19:")
print(source_model.model[19])

C3k2 indices: [2, 4, 6, 8, 13, 16, 19, 22]

Structure of module 19:
C3k2(
  (cv1): Conv(
    (conv): Conv2d(192, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (cv2): Conv(
    (conv): Conv2d(192, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (m): ModuleList(
    (0): C3k(
      (cv1): Conv(
        (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)

In [ ]:
conversion_report = {}

for index in indices:
    source = source_model.model[index]

    converted = SpikeC3k2(
        source,
        steps=2,
        beta=0.5,
        threshold=1.0,
        surrogate_slope=10.0,
    )

    original_state = source.state_dict()
    converted_state = converted.state_dict()

    assert set(original_state) == set(converted_state), (
        f"State-dict keys differ at module {index}"
    )

    for key in original_state:
        assert torch.equal(
            original_state[key],
            converted_state[key],
        ), f"Weight/buffer mismatch: module {index}, {key}"

    lif_count = sum(
        isinstance(module, LIF)
        for module in converted.modules()
    )

    assert lif_count > 0

    conversion_report[str(index)] = {
        "weight_transfer": "PASS",
        "lif_modules": lif_count,
        "parameters": sum(
            p.numel() for p in converted.parameters()
        ),
    }

print(json.dumps(conversion_report, indent=2))

{
  "2": {
    "weight_transfer": "PASS",
    "lif_modules": 2,
    "parameters": 6640
  },
  "4": {
    "weight_transfer": "PASS",
    "lif_modules": 2,
    "parameters": 26080
  },
  "6": {
    "weight_transfer": "PASS",
    "lif_modules": 4,
    "parameters": 87040
  },
  "8": {
    "weight_transfer": "PASS",
    "lif_modules": 4,
    "parameters": 346112
  },
  "13": {
    "weight_transfer": "PASS",
    "lif_modules": 4,
    "parameters": 119808
  },
  "16": {
    "weight_transfer": "PASS",
    "lif_modules": 4,
    "parameters": 34304
  },
  "19": {
    "weight_transfer": "PASS",
    "lif_modules": 4,
    "parameters": 95232
  },
  "22": {
    "weight_transfer": "PASS",
    "lif_modules": 2,
    "parameters": 463104
  }
}


In [ ]:
baseline_path = PROJECT / "runs/stage3_ann/ann_baseline.json"
baseline = json.loads(baseline_path.read_text())

shapes = baseline["c3k2_shapes"]

for index in indices:
    block = SpikeC3k2(
        source_model.model[index],
        steps=2,
    ).to(device).eval()

    input_shape = shapes[str(index)]["input_shape"]
    expected_shape = shapes[str(index)]["output_shape"]

    x = torch.randn(*input_shape, device=device)

    with torch.inference_mode():
        output = block(x)

    assert list(output.shape) == expected_shape, (
        f"Wrong output shape at module {index}"
    )
    assert torch.isfinite(output).all().item()

    conversion_report[str(index)]["shape_test"] = "PASS"

    print(
        f"Module {index}: "
        f"{list(x.shape)} -> {list(output.shape)} PASS"
    )

    del block, x, output

Module 2: [1, 32, 80, 80] -> [1, 64, 80, 80] PASS
Module 4: [1, 64, 40, 40] -> [1, 128, 40, 40] PASS
Module 6: [1, 128, 20, 20] -> [1, 128, 20, 20] PASS
Module 8: [1, 256, 10, 10] -> [1, 256, 10, 10] PASS
Module 13: [1, 384, 20, 20] -> [1, 128, 20, 20] PASS
Module 16: [1, 256, 40, 40] -> [1, 64, 40, 40] PASS
Module 19: [1, 192, 20, 20] -> [1, 128, 20, 20] PASS
Module 22: [1, 384, 10, 10] -> [1, 256, 10, 10] PASS


In [ ]:
amp_block = SpikeC3k2(
    source_model.model[19],
    steps=2,
).to(device).train()

amp_block.requires_grad_(True)
amp_block.zero_grad(set_to_none=True)

x_amp = torch.randn(
    2, 192, 20, 20,
    device=device,
)

with torch.autocast(device_type="cuda", dtype=torch.float16):
    amp_output = amp_block(x_amp)

amp_loss = (amp_output.float() - 0.25).square().mean()

assert torch.isfinite(amp_loss).item()
amp_loss.backward()

for name, parameter in amp_block.named_parameters():
    assert parameter.grad is not None, f"Missing gradient: {name}"
    assert torch.isfinite(parameter.grad).all().item(), (
        f"Non-finite AMP gradient: {name}"
    )

checks["mixed_precision_backward"] = "PASS"

torch.cuda.synchronize()

report = {
    "status": "PASS",
    "device": torch.cuda.get_device_name(0),
    "steps": 2,
    "outer_projections": "ANN",
    "inner_bottlenecks": "LIF",
    "readout": "mean_of_projected_temporal_features",
    "conversions": conversion_report,
    "checks": checks,
}

report_path = OUTPUT / "csp_report.json"

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print(json.dumps(report, indent=2))
print("\nSTAGE 6: PASS")
print("Report:", report_path)

{
  "status": "PASS",
  "device": "Tesla T4",
  "steps": 2,
  "outer_projections": "ANN",
  "inner_bottlenecks": "LIF",
  "readout": "mean_of_projected_temporal_features",
  "conversions": {
    "2": {
      "weight_transfer": "PASS",
      "lif_modules": 2,
      "parameters": 6640,
      "shape_test": "PASS"
    },
    "4": {
      "weight_transfer": "PASS",
      "lif_modules": 2,
      "parameters": 26080,
      "shape_test": "PASS"
    },
    "6": {
      "weight_transfer": "PASS",
      "lif_modules": 4,
      "parameters": 87040,
      "shape_test": "PASS"
    },
    "8": {
      "weight_transfer": "PASS",
      "lif_modules": 4,
      "parameters": 346112,
      "shape_test": "PASS"
    },
    "13": {
      "weight_transfer": "PASS",
      "lif_modules": 4,
      "parameters": 119808,
      "shape_test": "PASS"
    },
    "16": {
      "weight_transfer": "PASS",
      "lif_modules": 4,
      "parameters": 34304,
      "shape_test": "PASS"
    },
    "19": {
      "weight_transf

In [ ]:
%cd /content/yolo26_lif

import json
from pathlib import Path

import torch
from ultralytics import YOLO
from lif26.csp import SpikeC3k2


def complete_stage6():
    project = Path("/content/yolo26_lif")
    report_path = project / "runs/stage6_csp/csp_report.json"
    report = json.loads(report_path.read_text())

    # Check the previously recorded conversion results.
    expected_indices = {"2", "4", "6", "8", "13", "16", "19", "22"}

    assert set(report["conversions"]) == expected_indices

    for index, result in report["conversions"].items():
        assert result["weight_transfer"] == "PASS", index
        assert result["shape_test"] == "PASS", index

    stage6_checks = {
        "all_modules_weight_transfer": "PASS",
        "all_modules_output_shapes": "PASS",
    }

    assert torch.cuda.is_available()
    device = torch.device("cuda:0")
    torch.manual_seed(42)

    # Reload the original, unfused ANN checkpoint.
    original = YOLO(
        str(project / "weights/yolo26n.pt"),
        task="detect",
    ).model.cpu().eval()

    assert not hasattr(original, "snn_config")
    source = original.model[19]
    assert type(source).__name__ == "C3k2"

    # 1. State isolation and temporal averaging
    block = SpikeC3k2(source, steps=2).to(device).eval()
    x = torch.randn(1, 192, 20, 20, device=device)

    with torch.no_grad():
        output_a = block(x)
        output_b = block(x)

        torch.testing.assert_close(output_a, output_b)

        step1, state = block.forward_step(x, None)
        step2, state = block.forward_step(x, state)

        expected = (
            (step1.float() + step2.float()) / 2
        ).to(output_a.dtype)

        torch.testing.assert_close(
            output_a,
            expected,
            rtol=1e-4,
            atol=1e-5,
        )

    stage6_checks["independent_forward_calls"] = "PASS"
    stage6_checks["temporal_mean"] = "PASS"

    # 2. Backward and actual convolution weight updates
    train_block = SpikeC3k2(source, steps=2).to(device).train()
    train_block.requires_grad_(True)

    optimizer = torch.optim.SGD(
        train_block.parameters(),
        lr=0.01,
    )

    before = {
        name: parameter.detach().clone()
        for name, parameter in train_block.named_parameters()
        if parameter.ndim == 4
    }

    x_train = torch.randn(
        2, 192, 20, 20,
        device=device,
        requires_grad=True,
    )

    optimizer.zero_grad(set_to_none=True)

    output = train_block(x_train)
    loss = (output.float() - 0.25).square().mean()

    assert torch.isfinite(loss).item()
    loss.backward()

    assert x_train.grad is not None
    assert torch.isfinite(x_train.grad).all().item()
    assert x_train.grad.abs().sum().item() > 0

    for name, parameter in train_block.named_parameters():
        assert parameter.grad is not None, name
        assert torch.isfinite(parameter.grad).all().item(), name

        if name in before:
            assert parameter.grad.abs().sum().item() > 0, name

    optimizer.step()

    parameters = dict(train_block.named_parameters())

    for name, old_weight in before.items():
        assert not torch.equal(
            old_weight,
            parameters[name].detach(),
        ), f"Weight did not update: {name}"

    stage6_checks["module19_backward_and_optimizer"] = "PASS"

    # 3. Mixed-precision backward
    amp_block = SpikeC3k2(source, steps=2).to(device).train()
    amp_block.requires_grad_(True)
    amp_block.zero_grad(set_to_none=True)

    x_amp = torch.randn(2, 192, 20, 20, device=device)

    with torch.autocast(device_type="cuda", dtype=torch.float16):
        output_amp = amp_block(x_amp)

    assert output_amp.dtype == torch.float16
    assert torch.isfinite(output_amp).all().item()

    amp_loss = (output_amp.float() - 0.25).square().mean()
    amp_loss.backward()

    for name, parameter in amp_block.named_parameters():
        assert parameter.grad is not None, name
        assert torch.isfinite(parameter.grad).all().item(), name

        if parameter.ndim == 4:
            assert parameter.grad.abs().sum().item() > 0, name

    stage6_checks["mixed_precision_backward"] = "PASS"

    torch.cuda.synchronize()

    # Save only the relevant stage-six checks.
    report["checks"] = stage6_checks
    report["status"] = "PASS"

    report_path.write_text(
        json.dumps(report, indent=2),
        encoding="utf-8",
    )

    print(json.dumps(stage6_checks, indent=2))
    print("\nSTAGE 6 VERIFIED: PASS")
    print("Report:", report_path)


complete_stage6()

/content/yolo26_lif
{
  "all_modules_weight_transfer": "PASS",
  "all_modules_output_shapes": "PASS",
  "independent_forward_calls": "PASS",
  "temporal_mean": "PASS",
  "module19_backward_and_optimizer": "PASS",
  "mixed_precision_backward": "PASS"
}

STAGE 6 VERIFIED: PASS
Report: /content/yolo26_lif/runs/stage6_csp/csp_report.json


In [ ]:
import hashlib
import torch

from ultralytics import YOLO
from lif26.csp import SpikeC3k2
from lif26.neuron import LIF


SNN_CONFIG = {
    "index": 19,
    "steps": 2,
    "beta": 0.5,
    "threshold": 1.0,
    "surrogate_slope": 10.0,
}


def build_hybrid(config):
    yolo = YOLO(str(WEIGHTS), task="detect")

    assert not hasattr(yolo.model, "snn_config"), (
        "Use the original ANN checkpoint"
    )

    index = config["index"]
    source = yolo.model.model[index]

    assert type(source).__name__ == "C3k2"

    replacement = SpikeC3k2(
        source,
        steps=config["steps"],
        beta=config["beta"],
        threshold=config["threshold"],
        surrogate_slope=config["surrogate_slope"],
    )

    assert replacement.i == source.i
    assert replacement.f == source.f

    yolo.model.model[index] = replacement

    return yolo


hybrid = build_hybrid(SNN_CONFIG)
hybrid.model.to("cuda:0").eval()

converted_indices = [
    index
    for index, module in enumerate(hybrid.model.model)
    if isinstance(module, SpikeC3k2)
]

assert converted_indices == [19]

lif_count = sum(
    isinstance(module, LIF)
    for module in hybrid.model.modules()
)

assert lif_count == 4

print("Converted indices:", converted_indices)
print("LIF modules:", lif_count)
print("Temporal steps:", hybrid.model.model[19].steps)

Converted indices: [19]
LIF modules: 4
Temporal steps: 2


In [ ]:
def collect_tensors(value):
    if isinstance(value, torch.Tensor):
        return [value]

    if isinstance(value, dict):
        tensors = []
        for item in value.values():
            tensors.extend(collect_tensors(item))
        return tensors

    if isinstance(value, (tuple, list)):
        tensors = []
        for item in value:
            tensors.extend(collect_tensors(item))
        return tensors

    return []


torch.manual_seed(42)
x = torch.rand(1, 3, 320, 320, device="cuda:0")

observed_shapes = []


def module19_hook(module, inputs, output):
    observed_shapes.append(list(output.shape))


hook = hybrid.model.model[19].register_forward_hook(module19_hook)

try:
    with torch.inference_mode():
        result_a = hybrid.model(x)
        result_b = hybrid.model(x)
finally:
    hook.remove()

tensors_a = collect_tensors(result_a)
tensors_b = collect_tensors(result_b)

assert tensors_a
assert len(tensors_a) == len(tensors_b)

for a, b in zip(tensors_a, tensors_b):
    assert torch.isfinite(a).all().item()
    torch.testing.assert_close(a, b)

assert observed_shapes == [
    [1, 128, 20, 20],
    [1, 128, 20, 20],
]

checks7 = {
    "only_module19_converted": "PASS",
    "full_model_forward": "PASS",
    "module19_output_shape": "PASS",
    "independent_full_model_calls": "PASS",
}

print("Model output tensor shapes:")
for tensor in tensors_a:
    print(list(tensor.shape))

print("\nFull-model forward: PASS")

Model output tensor shapes:
[1, 84, 2100]
[1, 4, 2100]
[1, 80, 2100]
[1, 64, 40, 40]
[1, 128, 20, 20]
[1, 256, 10, 10]
[1, 4, 2100]
[1, 80, 2100]
[1, 64, 40, 40]
[1, 128, 20, 20]
[1, 256, 10, 10]

Full-model forward: PASS


In [ ]:
from copy import deepcopy

diagnostic_model = deepcopy(hybrid.model).train()
diagnostic_model.requires_grad_(True)
diagnostic_model.zero_grad(set_to_none=True)

x_train = torch.rand(
    2, 3, 320, 320,
    device="cuda:0",
    requires_grad=True,
)

raw_output = diagnostic_model(x_train)

differentiable_outputs = [
    tensor
    for tensor in collect_tensors(raw_output)
    if tensor.is_floating_point() and tensor.requires_grad
]

assert differentiable_outputs, (
    "No differentiable detection-head outputs found"
)

diagnostic_loss = sum(
    tensor.float().square().mean()
    for tensor in differentiable_outputs
)

assert torch.isfinite(diagnostic_loss).item()
diagnostic_loss.backward()

assert x_train.grad is not None
assert torch.isfinite(x_train.grad).all().item()
assert x_train.grad.abs().sum().item() > 0

# Every convolution in the converted block must receive a gradient.
checked_convolutions = []

for name, parameter in (
    diagnostic_model.model[19].named_parameters()
):
    if parameter.ndim != 4:
        continue

    assert parameter.grad is not None, name
    assert torch.isfinite(parameter.grad).all().item(), name
    assert parameter.grad.abs().sum().item() > 0, name

    checked_convolutions.append(name)

assert checked_convolutions

# At least one detection-head parameter must receive a nonzero gradient.
head_gradient_ok = any(
    parameter.grad is not None
    and torch.isfinite(parameter.grad).all().item()
    and parameter.grad.abs().sum().item() > 0
    for parameter in diagnostic_model.model[-1].parameters()
)

assert head_gradient_ok

checks7["full_model_backward"] = "PASS"

print("Diagnostic loss:", diagnostic_loss.item())
print("Checked module-19 convolutions:", checked_convolutions)
print("Full-model backward: PASS")

del diagnostic_model, raw_output, differentiable_outputs
del diagnostic_loss, x_train

Diagnostic loss: 710.5964965820312
Checked module-19 convolutions: ['cv1.conv.weight', 'cv2.conv.weight', 'm.0.cv1.conv.weight', 'm.0.cv2.conv.weight', 'm.0.cv3.conv.weight', 'm.0.m.0.cv1.conv.weight', 'm.0.m.0.cv2.conv.weight', 'm.0.m.1.cv1.conv.weight', 'm.0.m.1.cv2.conv.weight']
Full-model backward: PASS


In [ ]:
source_sha256 = hashlib.sha256(
    WEIGHTS.read_bytes()
).hexdigest()

checkpoint_path = OUTPUT / "hybrid19_state.pt"

torch.save(
    {
        "format_version": 1,
        "source_sha256": source_sha256,
        "snn_config": SNN_CONFIG,
        "state_dict": hybrid.model.state_dict(),
    },
    checkpoint_path,
)

print("Checkpoint:", checkpoint_path)

Checkpoint: /content/yolo26_lif/runs/stage6_csp/hybrid19_state.pt


In [ ]:
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

assert checkpoint["source_sha256"] == hashlib.sha256(
    WEIGHTS.read_bytes()
).hexdigest(), "Original ANN checkpoint has changed"

restored = build_hybrid(checkpoint["snn_config"])

restored.model.load_state_dict(
    checkpoint["state_dict"],
    strict=True,
)

restored.model.to("cuda:0").eval()

with torch.inference_mode():
    restored_result = restored.model(x)

restored_tensors = collect_tensors(restored_result)

assert len(restored_tensors) == len(tensors_a)

for reference, loaded in zip(tensors_a, restored_tensors):
    torch.testing.assert_close(reference, loaded)

checks7["checkpoint_round_trip"] = "PASS"

print("Checkpoint round trip: PASS")

del restored, restored_result, restored_tensors
del checkpoint

Checkpoint round trip: PASS


In [ ]:
validation_model = build_hybrid(SNN_CONFIG)

metrics = validation_model.val(
    data=str(DATA),
    split="val",
    imgsz=320,
    batch=2,
    device=0,
    workers=2,
    conf=0.001,
    iou=0.7,
    augment=False,
    plots=True,
    project=str(OUTPUT),
    name="validation",
    exist_ok=True,
)

evaluation = {
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "speed_ms_per_image": {
        key: float(value)
        for key, value in metrics.speed.items()
    },
    "save_dir": str(metrics.save_dir),
}

checks7["coco8_validation_completed"] = "PASS"

print(json.dumps(evaluation, indent=2))

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 133 layers, 2,409,060 parameters, 0 gradients, 5.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 930.1±300.0 MB/s, size: 54.0 KB)
val: Scanning /content/datasets/coco8/labels/val.cache... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 883.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.8it/s 0.5s
                   all          4         17      0.566        0.3      0.631      0.454
                person          3         10      0.195        0.3      0.208      0.121
                   dog          1          1          1          0      0.995      0.697
                 horse          1          2          0          0     0.0995     0.0182
              elephant          1          2      0.642        0.5      0.495      0.297
              umbrella          1          1    

In [ ]:
ann_report = json.loads(
    (PROJECT / "runs/stage3_ann/ann_baseline.json").read_text()
)

ann_eval = ann_report["evaluation"]

comparison = {}

for key in ["precision", "recall", "mAP50", "mAP50_95"]:
    comparison[key] = {
        "ANN": ann_eval[key],
        "Hybrid19": evaluation[key],
        "difference": evaluation[key] - ann_eval[key],
    }

report7 = {
    "status": "PASS",
    "scope": "Integration smoke test; no calibration or training",
    "converted_indices": [19],
    "lif_modules": lif_count,
    "snn_config": SNN_CONFIG,
    "source_sha256": source_sha256,
    "checkpoint": str(checkpoint_path),
    "checks": checks7,
    "evaluation": evaluation,
    "comparison": comparison,
}

report_path = OUTPUT / "integration_report.json"

report_path.write_text(
    json.dumps(report7, indent=2),
    encoding="utf-8",
)

print(json.dumps(comparison, indent=2))
print("\nSTAGE 7: PASS")
print("Report:", report_path)

{
  "precision": {
    "ANN": 0.6657676002260294,
    "Hybrid19": 0.5659574306401165,
    "difference": -0.09981016958591293
  },
  "recall": {
    "ANN": 0.6166666666666667,
    "Hybrid19": 0.3,
    "difference": -0.3166666666666667
  },
  "mAP50": {
    "ANN": 0.7799920510718633,
    "Hybrid19": 0.6312431043455634,
    "difference": -0.1487489467262999
  },
  "mAP50_95": {
    "ANN": 0.6052667022812892,
    "Hybrid19": 0.45413327695376887,
    "difference": -0.15113342532752033
  }
}

STAGE 7: PASS
Report: /content/yolo26_lif/runs/stage6_csp/integration_report.json


In [ ]:
%cd /content/yolo26_lif

import gc
import hashlib
import json
import statistics
from contextlib import nullcontext
from pathlib import Path

import torch
from torch import nn
from torch.nn.utils.fusion import fuse_conv_bn_eval
from ultralytics import YOLO

from lif26.csp import SpikeC3k2

PROJECT = Path("/content/yolo26_lif")
WEIGHTS = PROJECT / "weights/yolo26n.pt"
BENCH_OUTPUT = PROJECT / "runs/stage8_speed"

BENCH_OUTPUT.mkdir(parents=True, exist_ok=True)

assert WEIGHTS.is_file()
assert torch.cuda.is_available()

report_candidates = [
    PROJECT / "runs/stage7_hybrid19/integration_report.json",
    PROJECT / "runs/stage6_csp/integration_report.json",
]

integration_path = next(
    (p for p in report_candidates if p.is_file()),
    None,
)

assert integration_path is not None, "Stage 7 report not found"

integration = json.loads(integration_path.read_text())

required_checks = [
    "only_module19_converted",
    "full_model_forward",
    "module19_output_shape",
    "independent_full_model_calls",
    "full_model_backward",
    "checkpoint_round_trip",
    "coco8_validation_completed",
]

for name in required_checks:
    assert integration["checks"].get(name) == "PASS", name

CHECKPOINT = Path(integration["checkpoint"])
assert CHECKPOINT.is_file(), f"Missing checkpoint: {CHECKPOINT}"

payload = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=True,
)

assert payload["source_sha256"] == hashlib.sha256(
    WEIGHTS.read_bytes()
).hexdigest()

assert payload["snn_config"]["index"] == 19

print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint:", CHECKPOINT)
print("Stage 7 checks: PASS")

/content/yolo26_lif
GPU: Tesla T4
Checkpoint: /content/yolo26_lif/runs/stage6_csp/hybrid19_state.pt
Stage 7 checks: PASS


In [ ]:
def make_benchmark_model(name):
    model = YOLO(str(WEIGHTS), task="detect").model.cpu().eval()

    assert not hasattr(model, "snn_config")

    if name != "ANN":
        steps = {"Hybrid19_T1": 1, "Hybrid19_T2": 2}[name]
        config = payload["snn_config"]

        model.model[19] = SpikeC3k2(
            model.model[19],
            steps=steps,
            beta=config["beta"],
            threshold=config["threshold"],
            surrogate_slope=config["surrogate_slope"],
        )

        model.load_state_dict(
            payload["state_dict"],
            strict=True,
        )

    return model.to("cuda:0").eval()


def fuse_for_inference(model):
    fused_count = 0

    # Keep each module's normal forward API.
    # Its BatchNorm becomes Identity after fusion.
    for module in list(model.modules()):
        conv = getattr(module, "conv", None)
        bn = getattr(module, "bn", None)

        if isinstance(conv, nn.Conv2d) and isinstance(bn, nn.BatchNorm2d):
            assert not module.training
            module.conv = fuse_conv_bn_eval(conv, bn)
            module.bn = nn.Identity()
            fused_count += 1

    return fused_count


def collect_tensors(value):
    if isinstance(value, torch.Tensor):
        return [value]

    if isinstance(value, dict):
        values = value.values()
    elif isinstance(value, (tuple, list)):
        values = value
    else:
        return []

    result = []
    for item in values:
        result.extend(collect_tensors(item))
    return result

In [ ]:
def benchmark_forward(
    model,
    x,
    precision,
    warmup=30,
    repeats=100,
):
    context = (
        torch.autocast("cuda", dtype=torch.float16)
        if precision == "AMP_FP16"
        else nullcontext()
    )

    with torch.inference_mode(), context:
        # Validate before timing.
        sample = model(x)
        tensors = collect_tensors(sample)

        assert tensors
        for tensor in tensors:
            if tensor.is_floating_point():
                assert torch.isfinite(tensor).all().item()

        output_shapes = [list(t.shape) for t in tensors]
        del sample, tensors

        for _ in range(warmup):
            _ = model(x)

        del _
        torch.cuda.synchronize()

        baseline_memory = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()

        # Allocate events outside the timed loop.
        starts = [
            torch.cuda.Event(enable_timing=True)
            for _ in range(repeats)
        ]
        ends = [
            torch.cuda.Event(enable_timing=True)
            for _ in range(repeats)
        ]

        for index in range(repeats):
            starts[index].record()
            output = model(x)
            ends[index].record()
            del output

        torch.cuda.synchronize()

        times_ms = [
            start.elapsed_time(end)
            for start, end in zip(starts, ends)
        ]

        peak_memory = torch.cuda.max_memory_allocated()

    ordered = sorted(times_ms)
    median_ms = statistics.median(times_ms)

    return {
        "median_ms_per_batch": median_ms,
        "mean_ms_per_batch": statistics.mean(times_ms),
        "p90_ms_per_batch": ordered[
            max(0, int(0.9 * len(ordered)) - 1)
        ],
        "images_per_second": x.shape[0] * 1000.0 / median_ms,
        "baseline_allocated_mib": baseline_memory / 1024**2,
        "peak_allocated_mib": peak_memory / 1024**2,
        "additional_peak_mib": (
            peak_memory - baseline_memory
        ) / 1024**2,
        "output_shapes": output_shapes,
    }

In [ ]:
# Remove known old model/output references from earlier stages.
for variable_name in [
    "ann", "hybrid", "validation_model", "restored",
    "diagnostic_model", "source_yolo", "source_model",
    "block", "train_block", "amp_block",
    "conv", "downsample", "projection", "residual_test",
    "optimizer", "parameters", "parameter", "layer", "source",
    "x", "x_train", "x_amp", "current", "state",
    "result_a", "result_b", "tensors_a", "tensors_b",
    "output", "amp_output", "rate", "amp_rate",
    "loss", "amp_loss",
]:
    globals().pop(variable_name, None)

gc.collect()
torch.cuda.empty_cache()

# Fix backend settings across all cases.
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.manual_seed(42)

benchmark_input = torch.rand(
    1, 3, 320, 320,
    device="cuda:0",
)

benchmark_rows = []
fusion_audits = {}

for model_name in ["ANN", "Hybrid19_T1", "Hybrid19_T2"]:
    model = make_benchmark_model(model_name)

    # Record the effect of fusion on full-model outputs.
    with torch.inference_mode():
        before_fusion = [
            tensor.detach().clone()
            for tensor in collect_tensors(model(benchmark_input))
        ]

    fused_count = fuse_for_inference(model)

    with torch.inference_mode():
        after_fusion = collect_tensors(model(benchmark_input))

        assert len(before_fusion) == len(after_fusion)

        max_difference = 0.0
        outputs_close = True

        for before, after in zip(before_fusion, after_fusion):
            assert before.shape == after.shape
            assert torch.isfinite(after).all().item()

            max_difference = max(
                max_difference,
                (before.float() - after.float()).abs().max().item(),
            )

            outputs_close = outputs_close and torch.allclose(
                before.float(),
                after.float(),
                rtol=1e-3,
                atol=1e-4,
            )

    fusion_audits[model_name] = {
        "fused_conv_bn_pairs": fused_count,
        "outputs_close": outputs_close,
        "max_absolute_output_difference": max_difference,
    }

    del before_fusion, after_fusion, before, after

    print(
        f"\n{model_name}: fused={fused_count}, "
        f"fusion outputs close={outputs_close}"
    )

    for precision in ["FP32", "AMP_FP16"]:
        result = benchmark_forward(
            model,
            benchmark_input,
            precision,
        )

        row = {
            "model": model_name,
            "precision": precision,
            "fused_conv_bn_pairs": fused_count,
            **result,
        }

        benchmark_rows.append(row)

        print(
            f"{model_name:14s} {precision:8s} | "
            f"median={result['median_ms_per_batch']:.3f} ms | "
            f"P90={result['p90_ms_per_batch']:.3f} ms | "
            f"throughput={result['images_per_second']:.1f} images/s"
        )

    del model
    gc.collect()
    torch.cuda.empty_cache()


ANN: fused=114, fusion outputs close=True
ANN            FP32     | median=14.239 ms | P90=18.795 ms | throughput=70.2 images/s
ANN            AMP_FP16 | median=16.260 ms | P90=17.605 ms | throughput=61.5 images/s

Hybrid19_T1: fused=114, fusion outputs close=True
Hybrid19_T1    FP32     | median=12.153 ms | P90=13.263 ms | throughput=82.3 images/s
Hybrid19_T1    AMP_FP16 | median=17.631 ms | P90=18.713 ms | throughput=56.7 images/s

Hybrid19_T2: fused=114, fusion outputs close=True
Hybrid19_T2    FP32     | median=13.763 ms | P90=20.462 ms | throughput=72.7 images/s
Hybrid19_T2    AMP_FP16 | median=19.717 ms | P90=32.969 ms | throughput=50.7 images/s


In [ ]:
ann_times = {
    row["precision"]: row["median_ms_per_batch"]
    for row in benchmark_rows
    if row["model"] == "ANN"
}

for row in benchmark_rows:
    row["speedup_vs_ANN_same_precision"] = (
        ann_times[row["precision"]]
        / row["median_ms_per_batch"]
    )

benchmark_report = {
    "scope": "Model forward only; excludes preprocessing and postprocessing",
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "input_shape": list(benchmark_input.shape),
    "warmup": 30,
    "repeats": 100,
    "batch": 1,
    "imgsz": 320,
    "fusion_audits": fusion_audits,
    "results": benchmark_rows,
}

report_path = BENCH_OUTPUT / "speed_report.json"

report_path.write_text(
    json.dumps(benchmark_report, indent=2),
    encoding="utf-8",
)

print("\nResults:")
print("Model          Precision   Median(ms)  Speedup vs ANN")

for row in benchmark_rows:
    print(
        f"{row['model']:14s} "
        f"{row['precision']:10s} "
        f"{row['median_ms_per_batch']:10.3f} "
        f"{row['speedup_vs_ANN_same_precision']:10.3f}x"
    )

print("\nFusion audits:")
print(json.dumps(fusion_audits, indent=2))

print("\nSTAGE 8 BENCHMARK: COMPLETE")
print("Report:", report_path)


Results:
Model          Precision   Median(ms)  Speedup vs ANN
ANN            FP32           14.239      1.000x
ANN            AMP_FP16       16.260      1.000x
Hybrid19_T1    FP32           12.153      1.172x
Hybrid19_T1    AMP_FP16       17.631      0.922x
Hybrid19_T2    FP32           13.763      1.035x
Hybrid19_T2    AMP_FP16       19.717      0.825x

Fusion audits:
{
  "ANN": {
    "fused_conv_bn_pairs": 114,
    "outputs_close": true,
    "max_absolute_output_difference": 0.0005950927734375
  },
  "Hybrid19_T1": {
    "fused_conv_bn_pairs": 114,
    "outputs_close": true,
    "max_absolute_output_difference": 0.0006256103515625
  },
  "Hybrid19_T2": {
    "fused_conv_bn_pairs": 114,
    "outputs_close": true,
    "max_absolute_output_difference": 0.0011138916015625
  }
}

STAGE 8 BENCHMARK: COMPLETE
Report: /content/yolo26_lif/runs/stage8_speed/speed_report.json


In [ ]:
%cd /content/yolo26_lif

import gc
import json
import random
import statistics
import time
from pathlib import Path

import torch
from ultralytics import YOLO

assert callable(globals().get("make_benchmark_model"))
assert callable(globals().get("fuse_for_inference"))
assert callable(globals().get("collect_tensors"))

PROJECT = Path("/content/yolo26_lif")
STAGE9_OUTPUT = PROJECT / "runs/stage9_speed_verification"
STAGE9_OUTPUT.mkdir(parents=True, exist_ok=True)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.manual_seed(42)

stage9_input = torch.rand(
    1, 3, 320, 320,
    device="cuda:0",
    dtype=torch.float32,
)

print("GPU:", torch.cuda.get_device_name(0))

/content/yolo26_lif
GPU: Tesla T4


In [ ]:
def measure_serial_forward(model, x, warmup=30, repeats=75):
    gpu_times = []
    wall_times = []

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    with torch.inference_mode():
        sample = model(x)
        tensors = collect_tensors(sample)

        assert tensors
        for tensor in tensors:
            if tensor.is_floating_point():
                assert torch.isfinite(tensor).all().item()

        del sample, tensors

        for _ in range(warmup):
            warmup_output = model(x)

        del warmup_output
        torch.cuda.synchronize()

        for _ in range(repeats):
            wall_start = time.perf_counter()

            start_event.record()
            result = model(x)
            end_event.record()

            end_event.synchronize()

            wall_end = time.perf_counter()

            gpu_times.append(
                start_event.elapsed_time(end_event)
            )
            wall_times.append(
                (wall_end - wall_start) * 1000
            )

            del result

    return {
        "gpu_median_ms": statistics.median(gpu_times),
        "wall_median_ms": statistics.median(wall_times),
        "gpu_mean_ms": statistics.mean(gpu_times),
        "wall_mean_ms": statistics.mean(wall_times),
        "gpu_p90_ms": sorted(gpu_times)[
            int(0.9 * len(gpu_times)) - 1
        ],
        "wall_p90_ms": sorted(wall_times)[
            int(0.9 * len(wall_times)) - 1
        ],
    }

In [ ]:
orders = (
    [("ANN", "Hybrid19_T1")] * 3
    + [("Hybrid19_T1", "ANN")] * 3
)

random.Random(42).shuffle(orders)

round_results = []

for round_number, order in enumerate(orders, start=1):
    print(f"\nRound {round_number}: {order}")

    current_round = {
        "round": round_number,
        "order": list(order),
        "models": {},
    }

    for model_name in order:
        gc.collect()
        torch.cuda.empty_cache()

        model = make_benchmark_model(model_name)
        fused_count = fuse_for_inference(model)

        result = measure_serial_forward(
            model,
            stage9_input,
            warmup=30,
            repeats=75,
        )

        result["fused_conv_bn_pairs"] = fused_count
        current_round["models"][model_name] = result

        print(
            f"{model_name:14s} | "
            f"GPU={result['gpu_median_ms']:.3f} ms | "
            f"Wall={result['wall_median_ms']:.3f} ms"
        )

        del model
        gc.collect()
        torch.cuda.empty_cache()

    ann_result = current_round["models"]["ANN"]
    hybrid_result = current_round["models"]["Hybrid19_T1"]

    current_round["gpu_speedup"] = (
        ann_result["gpu_median_ms"]
        / hybrid_result["gpu_median_ms"]
    )

    current_round["wall_speedup"] = (
        ann_result["wall_median_ms"]
        / hybrid_result["wall_median_ms"]
    )

    round_results.append(current_round)


gpu_ratios = [r["gpu_speedup"] for r in round_results]
wall_ratios = [r["wall_speedup"] for r in round_results]

speed_summary = {
    "median_gpu_speedup": statistics.median(gpu_ratios),
    "median_wall_speedup": statistics.median(wall_ratios),
    "gpu_faster_rounds": sum(r > 1 for r in gpu_ratios),
    "wall_faster_rounds": sum(r > 1 for r in wall_ratios),
    "rounds": len(round_results),
    "gpu_speedup_range": [min(gpu_ratios), max(gpu_ratios)],
    "wall_speedup_range": [min(wall_ratios), max(wall_ratios)],
}

speed_report = {
    "scope": "Serial forward, resident GPU input, FP32, fused",
    "gpu": torch.cuda.get_device_name(0),
    "input_shape": list(stage9_input.shape),
    "warmup_per_case": 30,
    "repeats_per_case": 75,
    "round_results": round_results,
    "summary": speed_summary,
}

speed_path = STAGE9_OUTPUT / "repeated_speed_report.json"

speed_path.write_text(
    json.dumps(speed_report, indent=2),
    encoding="utf-8",
)

print("\nSpeed summary:")
print(json.dumps(speed_summary, indent=2))
print("Saved:", speed_path)


Round 1: ('Hybrid19_T1', 'ANN')
Hybrid19_T1    | GPU=12.213 ms | Wall=12.239 ms
ANN            | GPU=10.684 ms | Wall=10.708 ms

Round 2: ('ANN', 'Hybrid19_T1')
ANN            | GPU=10.683 ms | Wall=10.711 ms
Hybrid19_T1    | GPU=11.929 ms | Wall=11.956 ms

Round 3: ('ANN', 'Hybrid19_T1')
ANN            | GPU=15.813 ms | Wall=15.847 ms
Hybrid19_T1    | GPU=12.021 ms | Wall=12.050 ms

Round 4: ('Hybrid19_T1', 'ANN')
Hybrid19_T1    | GPU=12.283 ms | Wall=12.318 ms
ANN            | GPU=11.016 ms | Wall=11.042 ms

Round 5: ('ANN', 'Hybrid19_T1')
ANN            | GPU=10.994 ms | Wall=11.019 ms
Hybrid19_T1    | GPU=11.987 ms | Wall=12.015 ms

Round 6: ('Hybrid19_T1', 'ANN')
Hybrid19_T1    | GPU=16.344 ms | Wall=16.381 ms
ANN            | GPU=10.926 ms | Wall=10.952 ms

Speed summary:
{
  "median_gpu_speedup": 0.8962205560486336,
  "median_wall_speedup": 0.8961455665273926,
  "gpu_faster_rounds": 1,
  "wall_faster_rounds": 1,
  "rounds": 6,
  "gpu_speedup_range": [
    0.66847972152523,
    

In [ ]:
import torch
from torch import nn
from torch.nn.utils.fusion import fuse_conv_bn_eval

from ultralytics.nn.modules.conv import Conv as UltralyticsConv
from lif26.blocks import SpikeConv


def fuse_for_inference(model):
    model.eval()
    fused_count = 0

    for module in list(model.modules()):
        conv = getattr(module, "conv", None)
        bn = getattr(module, "bn", None)

        if not (
            isinstance(conv, nn.Conv2d)
            and isinstance(bn, nn.BatchNorm2d)
        ):
            continue

        if isinstance(module, UltralyticsConv):
            # Standard Ultralytics inference path:
            # fused Conv -> activation, without BatchNorm.
            module.conv = fuse_conv_bn_eval(conv, bn)
            delattr(module, "bn")
            module.forward = module.forward_fuse
            fused_count += 1

        elif isinstance(module, SpikeConv):
            # SpikeConv.forward_current still uses self.bn.
            module.conv = fuse_conv_bn_eval(conv, bn)
            module.bn = nn.Identity()
            fused_count += 1

        else:
            raise TypeError(
                f"Unsupported Conv/BN wrapper: {type(module).__name__}"
            )

    return fused_count


print("Fusion function corrected")

Fusion function corrected


In [ ]:
DATA = PROJECT / "configs/coco8_local.yaml"

In [ ]:
DATA = PROJECT / "configs/coco8_local.yaml"
assert DATA.is_file()

accuracy_results = {}

for model_name in ["ANN", "Hybrid19_T1"]:
    fused_model = make_benchmark_model(model_name)
    fuse_for_inference(fused_model)

    evaluator = YOLO(str(WEIGHTS), task="detect")
    evaluator.model = fused_model

    metrics = evaluator.val(
        data=str(DATA),
        split="val",
        imgsz=320,
        batch=2,
        device=0,
        workers=2,
        conf=0.001,
        iou=0.7,
        augment=False,
        plots=False,
        project=str(STAGE9_OUTPUT),
        name=f"{model_name}_validation",
        exist_ok=True,
    )

    accuracy_results[model_name] = {
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
        "save_dir": str(metrics.save_dir),
    }

    del evaluator, fused_model, metrics
    gc.collect()
    torch.cuda.empty_cache()


accuracy_difference = {
    key: (
        accuracy_results["Hybrid19_T1"][key]
        - accuracy_results["ANN"][key]
    )
    for key in ["precision", "recall", "mAP50", "mAP50_95"]
}

accuracy_report = {
    "scope": "COCO8 smoke evaluation; not full COCO accuracy",
    "results": accuracy_results,
    "difference_Hybrid_minus_ANN": accuracy_difference,
}

accuracy_path = STAGE9_OUTPUT / "t1_accuracy_report.json"

accuracy_path.write_text(
    json.dumps(accuracy_report, indent=2),
    encoding="utf-8",
)

print(json.dumps(accuracy_report, indent=2))
print("\nSTAGE 9: COMPLETE")
print("Accuracy report:", accuracy_path)

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 120 layers, 2,408,932 parameters, 0 gradients, 5.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1461.8±704.6 MB/s, size: 54.0 KB)
val: Scanning /content/datasets/coco8/labels/val.cache... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 1.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.8it/s 1.1s
                   all          4         17      0.666      0.617       0.78      0.605
                person          3         10      0.258        0.2      0.205       0.13
                   dog          1          1      0.675          1      0.995      0.697
                 horse          1          2      0.677          1      0.995      0.519
              elephant          1          2      0.817        0.5      0.495      0.396
              umbrella          1          1     

In [ ]:
%cd /content/yolo26_lif

import json
import random
import statistics
import time
from pathlib import Path

import torch
from torch import nn

assert callable(globals().get("make_benchmark_model"))
assert callable(globals().get("fuse_for_inference"))
assert callable(globals().get("collect_tensors"))
assert torch.cuda.is_available()

STAGE10_OUTPUT = Path(
    "/content/yolo26_lif/runs/stage10_compile"
)
STAGE10_OUTPUT.mkdir(parents=True, exist_ok=True)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.manual_seed(42)

compile_input = torch.rand(
    1, 3, 320, 320,
    device="cuda:0",
    dtype=torch.float32,
)

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

/content/yolo26_lif
PyTorch: 2.11.0+cu130
GPU: Tesla T4


In [ ]:
models10 = {}
setup_times10 = {}


def mark_iteration():
    torch.compiler.cudagraph_mark_step_begin()


for model_name in ["ANN", "Hybrid19_T1"]:
    eager_model = make_benchmark_model(model_name)
    fuse_for_inference(eager_model)

    eager_model.eval()
    eager_model.requires_grad_(False)

    # Ensure the corrected fusion function was used.
    from ultralytics.nn.modules.conv import Conv as UltralyticsConv

    for module in eager_model.modules():
        if isinstance(module, UltralyticsConv):
            assert not hasattr(module, "bn"), (
                "Use the corrected fuse_for_inference function"
            )

    # Initialize shape-dependent caches before compilation.
    with torch.inference_mode():
        for _ in range(3):
            temporary = eager_model(compile_input)
            del temporary

    torch.cuda.synchronize()

    print(f"\nCompiling {model_name}...", flush=True)
    start = time.perf_counter()

    compiled_model = torch.compile(
        eager_model,
        backend="inductor",
        mode="reduce-overhead",
        fullgraph=True,
        dynamic=False,
    )

    # Compilation is lazy: these calls actually trigger it.
    with torch.inference_mode():
        for _ in range(3):
            mark_iteration()
            temporary = compiled_model(compile_input)
            del temporary

    torch.cuda.synchronize()

    setup_times10[model_name] = time.perf_counter() - start

    models10[f"{model_name}_eager"] = eager_model
    models10[f"{model_name}_compiled"] = compiled_model

    print(
        f"Setup completed in "
        f"{setup_times10[model_name]:.1f} seconds",
        flush=True,
    )


Compiling ANN...


W1009 19:45:51.775000 1646 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


Setup completed in 55.6 seconds

Compiling Hybrid19_T1...
Setup completed in 24.1 seconds


In [ ]:
def snapshot_output(model, x):
    with torch.inference_mode():
        mark_iteration()
        output = model(x)

        tensors = collect_tensors(output)
        assert tensors, "No output tensors found"

        snapshot = [
            tensor.detach().cpu().clone()
            for tensor in tensors
        ]

    return snapshot


correctness10 = {}

for model_name in ["ANN", "Hybrid19_T1"]:
    largest_difference = 0.0

    for seed in [101, 202, 303]:
        torch.manual_seed(seed)

        test_input = torch.rand_like(compile_input)

        reference = snapshot_output(
            models10[f"{model_name}_eager"],
            test_input,
        )

        compiled = snapshot_output(
            models10[f"{model_name}_compiled"],
            test_input,
        )

        assert len(reference) == len(compiled)

        for expected, actual in zip(reference, compiled):
            assert expected.shape == actual.shape
            assert expected.dtype == actual.dtype
            assert torch.isfinite(actual).all().item()

            torch.testing.assert_close(
                actual,
                expected,
                rtol=1e-3,
                atol=1e-4,
            )

            largest_difference = max(
                largest_difference,
                (actual.float() - expected.float())
                .abs().max().item(),
            )

    correctness10[model_name] = {
        "status": "PASS",
        "tested_inputs": 3,
        "max_absolute_difference": largest_difference,
    }

print(json.dumps(correctness10, indent=2))

{
  "ANN": {
    "status": "PASS",
    "tested_inputs": 3,
    "max_absolute_difference": 0.000885009765625
  },
  "Hybrid19_T1": {
    "status": "PASS",
    "tested_inputs": 3,
    "max_absolute_difference": 0.000762939453125
  }
}


In [ ]:
def measure_compiled_case(model, x, warmup=20, repeats=75):
    gpu_times = []
    wall_times = []

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    with torch.inference_mode():
        for _ in range(warmup):
            mark_iteration()
            output = model(x)
            del output

        torch.cuda.synchronize()

        for _ in range(repeats):
            wall_start = time.perf_counter()

            mark_iteration()
            start_event.record()

            output = model(x)

            end_event.record()
            end_event.synchronize()

            wall_end = time.perf_counter()

            gpu_times.append(
                start_event.elapsed_time(end_event)
            )
            wall_times.append(
                (wall_end - wall_start) * 1000
            )

            del output

    return {
        "gpu_median_ms": statistics.median(gpu_times),
        "wall_median_ms": statistics.median(wall_times),
        "wall_p90_ms": sorted(wall_times)[
            int(0.9 * len(wall_times)) - 1
        ],
    }

In [ ]:
case_names = [
    "ANN_eager",
    "ANN_compiled",
    "Hybrid19_T1_eager",
    "Hybrid19_T1_compiled",
]

orders = [
    case_names[offset:] + case_names[:offset]
    for offset in range(len(case_names))
]

random.Random(42).shuffle(orders)

compile_rounds = []

for round_number, order in enumerate(orders, start=1):
    print(f"\nRound {round_number}", flush=True)

    round_record = {
        "round": round_number,
        "order": order,
        "results": {},
    }

    for case_name in order:
        result = measure_compiled_case(
            models10[case_name],
            compile_input,
        )

        round_record["results"][case_name] = result

        print(
            f"{case_name:24s} | "
            f"GPU={result['gpu_median_ms']:.3f} ms | "
            f"Wall={result['wall_median_ms']:.3f} ms",
            flush=True,
        )

    compile_rounds.append(round_record)


Round 1
Hybrid19_T1_eager        | GPU=22.007 ms | Wall=22.079 ms
Hybrid19_T1_compiled     | GPU=3.486 ms | Wall=3.587 ms
ANN_eager                | GPU=11.243 ms | Wall=11.282 ms
ANN_compiled             | GPU=3.706 ms | Wall=3.779 ms

Round 2
ANN_compiled             | GPU=3.645 ms | Wall=3.741 ms
Hybrid19_T1_eager        | GPU=11.978 ms | Wall=12.015 ms
Hybrid19_T1_compiled     | GPU=3.573 ms | Wall=3.637 ms
ANN_eager                | GPU=14.460 ms | Wall=14.514 ms

Round 3
Hybrid19_T1_compiled     | GPU=3.651 ms | Wall=3.776 ms
ANN_eager                | GPU=11.592 ms | Wall=11.641 ms
ANN_compiled             | GPU=3.690 ms | Wall=3.765 ms
Hybrid19_T1_eager        | GPU=12.654 ms | Wall=12.695 ms

Round 4
ANN_eager                | GPU=11.430 ms | Wall=11.472 ms
ANN_compiled             | GPU=3.664 ms | Wall=3.764 ms
Hybrid19_T1_eager        | GPU=12.027 ms | Wall=12.062 ms
Hybrid19_T1_compiled     | GPU=3.451 ms | Wall=3.548 ms


In [ ]:
median_wall = {
    name: statistics.median(
        round_record["results"][name]["wall_median_ms"]
        for round_record in compile_rounds
    )
    for name in case_names
}


def paired_speedup(reference_name, candidate_name):
    ratios = [
        r["results"][reference_name]["wall_median_ms"]
        / r["results"][candidate_name]["wall_median_ms"]
        for r in compile_rounds
    ]

    return {
        "median_speedup": statistics.median(ratios),
        "faster_rounds": sum(value > 1 for value in ratios),
        "total_rounds": len(ratios),
        "range": [min(ratios), max(ratios)],
    }


summary10 = {
    "wall_median_ms": median_wall,
    "compile_effect_ANN": paired_speedup(
        "ANN_eager",
        "ANN_compiled",
    ),
    "compile_effect_Hybrid": paired_speedup(
        "Hybrid19_T1_eager",
        "Hybrid19_T1_compiled",
    ),
    "compiled_Hybrid_vs_compiled_ANN": paired_speedup(
        "ANN_compiled",
        "Hybrid19_T1_compiled",
    ),
}

report10 = {
    "scope": "FP32 fused forward, batch=1, 320x320",
    "gpu": torch.cuda.get_device_name(0),
    "torch_version": str(torch.__version__),
    "compile_mode": "reduce-overhead",
    "fullgraph": True,
    "setup_seconds": setup_times10,
    "correctness": correctness10,
    "rounds": compile_rounds,
    "summary": summary10,
}

report_path = STAGE10_OUTPUT / "compile_report.json"

report_path.write_text(
    json.dumps(report10, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary10, indent=2))
print("\nSTAGE 10: COMPLETE")
print("Report:", report_path)

{
  "wall_median_ms": {
    "ANN_eager": 11.556308499848456,
    "ANN_compiled": 3.764498000009553,
    "Hybrid19_T1_eager": 12.378767999734919,
    "Hybrid19_T1_compiled": 3.6124670000390324
  },
  "compile_effect_ANN": {
    "median_speedup": 3.0698098169605883,
    "faster_rounds": 4,
    "total_rounds": 4,
    "range": [
      2.985758776006722,
      3.880218696863092
    ]
  },
  "compile_effect_Hybrid": {
    "median_speedup": 3.3812156367477626,
    "faster_rounds": 4,
    "total_rounds": 4,
    "range": [
      3.303134721719107,
      6.154326906557542
    ]
  },
  "compiled_Hybrid_vs_compiled_ANN": {
    "median_speedup": 1.040814857079356,
    "faster_rounds": 3,
    "total_rounds": 4,
    "range": [
      0.9972457446311488,
      1.06089684772747
    ]
  }
}

STAGE 10: COMPLETE
Report: /content/yolo26_lif/runs/stage10_compile/compile_report.json


In [ ]:
%cd /content/yolo26_lif

import json
from pathlib import Path

import cv2
import numpy as np
import torch

from ultralytics.data.augment import LetterBox
from ultralytics.data.utils import check_det_dataset

PROJECT = Path("/content/yolo26_lif")
STAGE11_OUTPUT = PROJECT / "runs/stage11_real_image_check"
STAGE11_OUTPUT.mkdir(parents=True, exist_ok=True)

assert "models10" in globals(), (
    "Compiled models are missing. Re-run stage 10 model setup."
)

required_models = [
    "ANN_eager",
    "ANN_compiled",
    "Hybrid19_T1_eager",
    "Hybrid19_T1_compiled",
]

for name in required_models:
    assert name in models10, f"Missing model: {name}"

assert callable(globals().get("collect_tensors"))
assert callable(globals().get("snapshot_output"))

dataset = check_det_dataset(
    str(PROJECT / "configs/coco8_local.yaml")
)

val_dir = Path(dataset["val"])
assert val_dir.is_dir()

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_paths = sorted(
    path
    for path in val_dir.rglob("*")
    if path.is_file() and path.suffix.lower() in extensions
)

assert len(image_paths) == 4, (
    f"Expected 4 COCO8 validation images; found {len(image_paths)}"
)

print("Validation images:")
for path in image_paths:
    print(path.name)

/content/yolo26_lif
Validation images:
000000000036.jpg
000000000042.jpg
000000000049.jpg
000000000061.jpg


In [ ]:
letterbox = LetterBox(
    new_shape=(320, 320),
    auto=False,
    scale_fill=False,
    scaleup=True,
    stride=32,
)


def prepare_real_image(path):
    image_bgr = cv2.imread(str(path))
    assert image_bgr is not None, f"Cannot read: {path}"

    resized_bgr = letterbox(image=image_bgr)

    # BGR HWC -> RGB CHW
    image_rgb_chw = resized_bgr[:, :, ::-1].transpose(2, 0, 1)
    image_rgb_chw = np.ascontiguousarray(image_rgb_chw)

    tensor = torch.from_numpy(image_rgb_chw)
    tensor = tensor.to(
        device="cuda:0",
        dtype=torch.float32,
    )

    tensor = tensor.unsqueeze(0) / 255.0

    assert tensor.shape == (1, 3, 320, 320)
    assert torch.isfinite(tensor).all().item()

    return tensor

In [ ]:
def compare_output_snapshots(reference, candidate):
    if len(reference) != len(candidate):
        return {
            "passed": False,
            "reason": "Different number of output tensors",
            "reference_count": len(reference),
            "candidate_count": len(candidate),
        }

    tensor_reports = []

    for index, (expected, actual) in enumerate(
        zip(reference, candidate)
    ):
        same_shape = expected.shape == actual.shape
        same_dtype = expected.dtype == actual.dtype

        entry = {
            "tensor_index": index,
            "same_shape": same_shape,
            "same_dtype": same_dtype,
        }

        if not same_shape or not same_dtype:
            entry["passed"] = False
            tensor_reports.append(entry)
            continue

        finite = (
            torch.isfinite(expected).all().item()
            and torch.isfinite(actual).all().item()
        )

        if expected.is_floating_point():
            difference = (
                expected.float() - actual.float()
            ).abs()

            close = torch.isclose(
                expected,
                actual,
                rtol=1e-3,
                atol=1e-4,
            )

            entry.update({
                "max_absolute_difference": (
                    difference.max().item()
                    if difference.numel() else 0.0
                ),
                "elements_outside_tolerance": (
                    (~close).sum().item()
                ),
                "passed": bool(finite and close.all().item()),
            })

        else:
            entry["passed"] = bool(
                finite and torch.equal(expected, actual)
            )

        tensor_reports.append(entry)

    return {
        "passed": all(item["passed"] for item in tensor_reports),
        "tensors": tensor_reports,
    }


real_image_checks = []

for image_path in image_paths:
    image_tensor = prepare_real_image(image_path)

    for model_name in ["ANN", "Hybrid19_T1"]:
        # snapshot_output copies outputs to CPU, avoiding reuse
        # of CUDA Graph output storage between calls.
        reference = snapshot_output(
            models10[f"{model_name}_eager"],
            image_tensor,
        )

        compiled = snapshot_output(
            models10[f"{model_name}_compiled"],
            image_tensor,
        )

        comparison = compare_output_snapshots(
            reference,
            compiled,
        )

        record = {
            "image": image_path.name,
            "model": model_name,
            **comparison,
        }

        real_image_checks.append(record)

        status = "PASS" if comparison["passed"] else "FAIL"
        print(f"{model_name:14s} | {image_path.name} | {status}")

    del image_tensor

ANN            | 000000000036.jpg | PASS
Hybrid19_T1    | 000000000036.jpg | PASS
ANN            | 000000000042.jpg | PASS
Hybrid19_T1    | 000000000042.jpg | PASS
ANN            | 000000000049.jpg | PASS
Hybrid19_T1    | 000000000049.jpg | PASS
ANN            | 000000000061.jpg | PASS
Hybrid19_T1    | 000000000061.jpg | PASS


In [ ]:
state_checks = {}

image_a = prepare_real_image(image_paths[0])
image_b = prepare_real_image(image_paths[1])

for model_name in ["ANN", "Hybrid19_T1"]:
    compiled_model = models10[f"{model_name}_compiled"]

    first_a = snapshot_output(compiled_model, image_a)
    _ = snapshot_output(compiled_model, image_b)
    second_a = snapshot_output(compiled_model, image_a)

    state_checks[model_name] = compare_output_snapshots(
        first_a,
        second_a,
    )

    status = (
        "PASS"
        if state_checks[model_name]["passed"]
        else "FAIL"
    )

    print(f"{model_name}: A -> B -> A independence: {status}")

del image_a, image_b

ANN: A -> B -> A independence: PASS
Hybrid19_T1: A -> B -> A independence: PASS


In [ ]:
all_passed = (
    all(item["passed"] for item in real_image_checks)
    and all(item["passed"] for item in state_checks.values())
)

report11 = {
    "status": "PASS" if all_passed else "FAIL",
    "scope": "Compiled vs eager output checks on COCO8 images",
    "input_shape": [1, 3, 320, 320],
    "precision": "FP32",
    "rtol": 1e-3,
    "atol": 1e-4,
    "image_checks": real_image_checks,
    "state_independence": state_checks,
}

report_path = STAGE11_OUTPUT / "real_image_report.json"

report_path.write_text(
    json.dumps(report11, indent=2),
    encoding="utf-8",
)

print("\nImage/model comparisons:", len(real_image_checks))
print(
    "Passed:",
    sum(item["passed"] for item in real_image_checks),
)
print("STAGE 11:", report11["status"])
print("Report:", report_path)

if not all_passed:
    print("\nFailed checks:")
    for item in real_image_checks:
        if not item["passed"]:
            print(json.dumps(item, indent=2))


Image/model comparisons: 8
Passed: 8
STAGE 11: PASS
Report: /content/yolo26_lif/runs/stage11_real_image_check/real_image_report.json


In [83]:
%cd /content/yolo26_lif

import json
from pathlib import Path

PROJECT = Path("/content/yolo26_lif")
STAGE12_OUTPUT = PROJECT / "runs/stage12_fast_t1"
STAGE12_OUTPUT.mkdir(parents=True, exist_ok=True)

stage11 = json.loads(
    (
        PROJECT
        / "runs/stage11_real_image_check/real_image_report.json"
    ).read_text()
)

assert stage11["status"] == "PASS"

assert "models10" in globals()
assert "Hybrid19_T1_eager" in models10
assert "Hybrid19_T1_compiled" in models10
assert "ANN_compiled" in models10

for name in [
    "snapshot_output",
    "compare_output_snapshots",
    "prepare_real_image",
    "measure_compiled_case",
]:
    assert callable(globals().get(name)), f"Missing function: {name}"

assert "image_paths" in globals()
assert len(image_paths) == 4

print("Stage 12 prerequisites: PASS")

/content/yolo26_lif
Stage 12 prerequisites: PASS


In [84]:
%%writefile /content/yolo26_lif/lif26/t1_inference.py
from copy import deepcopy

import torch
from torch import nn

from .blocks import SpikeConv, SpikeBottleneck
from .csp import SpikeC3k, SpikeC3k2, SpikeSequence


def binary_threshold(current, threshold):
    # Preserve the original FP32 thresholding behavior.
    return (current.float() >= threshold).to(current.dtype)


class T1SpikeConv(nn.Module):
    def __init__(self, source):
        super().__init__()
        self.conv = source.conv
        self.bn = source.bn
        self.threshold = float(source.lif.threshold)

    def forward_current(self, x):
        return self.bn(self.conv(x))

    def forward(self, x):
        return binary_threshold(
            self.forward_current(x),
            self.threshold,
        )


class T1SpikeBottleneck(nn.Module):
    def __init__(self, source):
        super().__init__()
        self.cv1 = T1SpikeConv(source.cv1)
        self.cv2 = T1SpikeConv(source.cv2)
        self.add = bool(source.add)

    def forward(self, x):
        spike1 = self.cv1(x)
        current2 = self.cv2.forward_current(spike1)

        if self.add:
            current2 = current2.float() + x.float()

        spike2 = binary_threshold(
            current2,
            self.cv2.threshold,
        )

        return spike2.to(spike1.dtype)


class T1SpikeC3k(nn.Module):
    def __init__(self, source):
        super().__init__()

        self.cv1 = source.cv1
        self.cv2 = source.cv2
        self.cv3 = source.cv3

        self.m = nn.Sequential(
            *(convert_child(child) for child in source.m)
        )

    def forward(self, x):
        main = self.m(self.cv1(x))
        bypass = self.cv2(x)

        return self.cv3(
            torch.cat((main, bypass), dim=1)
        )


def convert_child(source):
    if isinstance(source, SpikeBottleneck):
        return T1SpikeBottleneck(source)

    if isinstance(source, SpikeC3k):
        return T1SpikeC3k(source)

    if isinstance(source, (SpikeSequence, nn.Sequential)):
        return nn.Sequential(
            *(convert_child(child) for child in source)
        )

    if type(source).__name__ == "PSABlock":
        return source

    raise TypeError(
        f"Unsupported child: {type(source).__name__}"
    )


class T1SpikeC3k2(nn.Module):
    def __init__(self, source):
        super().__init__()

        if not isinstance(source, SpikeC3k2):
            raise TypeError("Expected SpikeC3k2")
        if source.steps != 1:
            raise ValueError("This inference path supports only T=1")

        self.cv1 = source.cv1
        self.cv2 = source.cv2
        self.c = source.c

        self.m = nn.ModuleList(
            convert_child(child)
            for child in source.m
        )

        for name in ("i", "f", "type", "np"):
            if hasattr(source, name):
                setattr(self, name, deepcopy(getattr(source, name)))

        self.eval()

    def forward(self, x):
        if self.training or torch.is_grad_enabled():
            raise RuntimeError(
                "Use eval() and torch.inference_mode() "
                "for the T1 inference model"
            )

        features = list(self.cv1(x).chunk(2, dim=1))

        for layer in self.m:
            features.append(layer(features[-1]))

        return self.cv2(torch.cat(features, dim=1))


def build_fast_t1(reference_model, index=19):
    # Work on an independent copy of the already fused reference.
    result = deepcopy(reference_model).eval()

    source = result.model[index]

    if not isinstance(source, SpikeC3k2):
        raise TypeError("The selected block is not SpikeC3k2")
    if source.steps != 1:
        raise ValueError("The selected block must use T=1")

    result.model[index] = T1SpikeC3k2(source)

    result.eval()
    result.requires_grad_(False)

    return result

Writing /content/yolo26_lif/lif26/t1_inference.py


In [85]:
import torch

from lif26.t1_inference import build_fast_t1

reference_t1 = models10["Hybrid19_T1_eager"]

assert reference_t1.model[19].steps == 1

fast_t1 = build_fast_t1(reference_t1).to("cuda:0").eval()

reference_state = reference_t1.state_dict()
fast_state = fast_t1.state_dict()

assert set(reference_state) == set(fast_state)

for name in reference_state:
    assert torch.equal(
        reference_state[name],
        fast_state[name],
    ), f"Weight or buffer changed: {name}"

del reference_state, fast_state

print("All weights and buffers preserved: PASS")

All weights and buffers preserved: PASS


In [86]:
fast_eager_checks = []

for image_path in image_paths:
    image_tensor = prepare_real_image(image_path)

    reference_output = snapshot_output(
        reference_t1,
        image_tensor,
    )

    fast_output = snapshot_output(
        fast_t1,
        image_tensor,
    )

    comparison = compare_output_snapshots(
        reference_output,
        fast_output,
    )

    fast_eager_checks.append({
        "image": image_path.name,
        **comparison,
    })

    status = "PASS" if comparison["passed"] else "FAIL"
    print(image_path.name, status)

eager_report_path = STAGE12_OUTPUT / "eager_equivalence.json"

eager_report_path.write_text(
    json.dumps(fast_eager_checks, indent=2),
    encoding="utf-8",
)

assert all(
    result["passed"] for result in fast_eager_checks
), "Output mismatch; inspect eager_equivalence.json"

print("Fast T1 eager equivalence: PASS")

000000000036.jpg PASS
000000000042.jpg PASS
000000000049.jpg PASS
000000000061.jpg PASS
Fast T1 eager equivalence: PASS


In [87]:
import time

torch.manual_seed(42)

stage12_input = torch.rand(
    1, 3, 320, 320,
    device="cuda:0",
    dtype=torch.float32,
)

# Initialize shape-dependent caches.
with torch.inference_mode():
    for _ in range(3):
        temporary = fast_t1(stage12_input)
        del temporary

torch.cuda.synchronize()

setup_start = time.perf_counter()

fast_t1_compiled = torch.compile(
    fast_t1,
    backend="inductor",
    mode="reduce-overhead",
    fullgraph=True,
    dynamic=False,
)

with torch.inference_mode():
    for _ in range(3):
        torch.compiler.cudagraph_mark_step_begin()
        temporary = fast_t1_compiled(stage12_input)
        del temporary

torch.cuda.synchronize()

fast_compile_setup_seconds = time.perf_counter() - setup_start

print(
    "Compile/setup time:",
    round(fast_compile_setup_seconds, 2),
    "seconds",
)

Compile/setup time: 16.79 seconds


In [88]:
fast_compiled_checks = []

for image_path in image_paths:
    image_tensor = prepare_real_image(image_path)

    reference_output = snapshot_output(
        reference_t1,
        image_tensor,
    )

    compiled_output = snapshot_output(
        fast_t1_compiled,
        image_tensor,
    )

    comparison = compare_output_snapshots(
        reference_output,
        compiled_output,
    )

    fast_compiled_checks.append({
        "image": image_path.name,
        **comparison,
    })

    status = "PASS" if comparison["passed"] else "FAIL"
    print(image_path.name, status)

compiled_report_path = (
    STAGE12_OUTPUT / "compiled_equivalence.json"
)

compiled_report_path.write_text(
    json.dumps(fast_compiled_checks, indent=2),
    encoding="utf-8",
)

assert all(
    result["passed"] for result in fast_compiled_checks
), "Compiled output mismatch; inspect compiled_equivalence.json"

print("Fast T1 compiled equivalence: PASS")

000000000036.jpg PASS
000000000042.jpg PASS
000000000049.jpg PASS
000000000061.jpg PASS
Fast T1 compiled equivalence: PASS


In [89]:
import itertools
import random
import statistics

stage12_models = {
    "ANN_compiled": models10["ANN_compiled"],
    "Hybrid_original_compiled": models10["Hybrid19_T1_compiled"],
    "Hybrid_fastT1_compiled": fast_t1_compiled,
}

orders12 = list(itertools.permutations(stage12_models))
random.Random(42).shuffle(orders12)

rounds12 = []

for round_number, order in enumerate(orders12, start=1):
    print(f"\nRound {round_number}", flush=True)

    record = {
        "round": round_number,
        "order": list(order),
        "results": {},
    }

    for name in order:
        result = measure_compiled_case(
            stage12_models[name],
            stage12_input,
            warmup=20,
            repeats=75,
        )

        record["results"][name] = result

        print(
            f"{name:28s} | "
            f"Wall={result['wall_median_ms']:.3f} ms",
            flush=True,
        )

    rounds12.append(record)


Round 1
Hybrid_original_compiled     | Wall=4.758 ms
Hybrid_fastT1_compiled       | Wall=3.816 ms
ANN_compiled                 | Wall=4.095 ms

Round 2
ANN_compiled                 | Wall=4.054 ms
Hybrid_fastT1_compiled       | Wall=3.525 ms
Hybrid_original_compiled     | Wall=3.719 ms

Round 3
Hybrid_original_compiled     | Wall=3.733 ms
ANN_compiled                 | Wall=3.749 ms
Hybrid_fastT1_compiled       | Wall=3.570 ms

Round 4
Hybrid_fastT1_compiled       | Wall=3.536 ms
ANN_compiled                 | Wall=3.738 ms
Hybrid_original_compiled     | Wall=3.748 ms

Round 5
ANN_compiled                 | Wall=3.730 ms
Hybrid_original_compiled     | Wall=3.721 ms
Hybrid_fastT1_compiled       | Wall=3.560 ms

Round 6
Hybrid_fastT1_compiled       | Wall=3.549 ms
Hybrid_original_compiled     | Wall=3.735 ms
ANN_compiled                 | Wall=3.736 ms


In [90]:
def compare_stage12(reference_name, candidate_name):
    ratios = [
        result["results"][reference_name]["wall_median_ms"]
        / result["results"][candidate_name]["wall_median_ms"]
        for result in rounds12
    ]

    return {
        "median_speedup": statistics.median(ratios),
        "faster_rounds": sum(ratio > 1 for ratio in ratios),
        "total_rounds": len(ratios),
        "range": [min(ratios), max(ratios)],
    }


summary12 = {
    "wall_median_ms": {
        name: statistics.median(
            result["results"][name]["wall_median_ms"]
            for result in rounds12
        )
        for name in stage12_models
    },
    "fastT1_vs_original_Hybrid": compare_stage12(
        "Hybrid_original_compiled",
        "Hybrid_fastT1_compiled",
    ),
    "fastT1_vs_ANN": compare_stage12(
        "ANN_compiled",
        "Hybrid_fastT1_compiled",
    ),
}

report12 = {
    "scope": "T=1 inference specialization, FP32, fused, compiled",
    "input_shape": list(stage12_input.shape),
    "weights_preserved": True,
    "eager_equivalence": fast_eager_checks,
    "compiled_equivalence": fast_compiled_checks,
    "compile_setup_seconds": fast_compile_setup_seconds,
    "rounds": rounds12,
    "summary": summary12,
}

report_path = STAGE12_OUTPUT / "fast_t1_report.json"

report_path.write_text(
    json.dumps(report12, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary12, indent=2))
print("\nSTAGE 12: COMPLETE")
print("Report:", report_path)

{
  "wall_median_ms": {
    "ANN_compiled": 3.743659000065236,
    "Hybrid_original_compiled": 3.7336749996939034,
    "Hybrid_fastT1_compiled": 3.554190499471588
  },
  "fastT1_vs_original_Hybrid": {
    "median_speedup": 1.053621170436637,
    "faster_rounds": 6,
    "total_rounds": 6,
    "range": [
      1.0452793180430109,
      1.2471087296193366
    ]
  },
  "fastT1_vs_ANN": {
    "median_speedup": 1.0550290133427507,
    "faster_rounds": 6,
    "total_rounds": 6,
    "range": [
      1.048022378478318,
      1.1498596886235868
    ]
  }
}

STAGE 12: COMPLETE
Report: /content/yolo26_lif/runs/stage12_fast_t1/fast_t1_report.json
